# Transformer

### 서울

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings
import math

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리 (결측치 행 삭제)
def load_and_prepare():
    data_path = os.path.join(DATA_DIR, 'sido_서울특별시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age'] # 학습에 사용할 컬럼 정의
    
    # 결측치가 있는 행 삭제 (실거래 데이터만 남김)
    df = df.dropna(subset=FEATURES)
    
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id']) # 지역명(region_id)을 인공지능이 이해할 수 있는 숫자(0, 1, 2...)로 변환
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

# 아파트 단지(sample_id)별로 그룹화하여 처리
for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue # 데이터가 12개 이하인 단지는 학습 불가하므로 제외
    
    sample_windows = []
    values = group[FEATURES].values # 가격, 면적 등 수치 데이터
    reg_idxs = group['region_idx'].values # 해당 단지의 지역 번호
    
    # 슬라이딩 윈도우: 1~12월로 13월 예측, 2~13월로 14월 예측...
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], # 입력: 12개월치 시계열 특징
            'region_idx': reg_idxs[i+WINDOW], # 입력: 지역 정보 (Embedding용)
            'y': values[i+WINDOW, 0] # 정답: 다음 달의 가격(price)
        })
    
    # [데이터 분할] 한 단지의 전체 기록 중 앞 70%는 학습, 10%는 검증, 20%는 테스트용
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
n = len(train_df) + len(val_df) + len(test_df)
if n == 0:
    print("경고: 윈도우가 생성되지 않았습니다.")

# Train 데이터의 평균과 표준편차를 기준으로 전체 데이터 스케일링 (표준화)
scaler = StandardScaler()
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class MultiModalDataset(Dataset):
    def __init__(self, data_df, scaler, n_features):
        # 3차원 배열 생성: (샘플 수, 12개월, 특징 수)
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        # 입력 시퀀스 데이터 스케일링 및 텐서 변환
        self.x_seq = torch.FloatTensor(scaler.transform(x_raw.reshape(-1, F)).reshape(N, W, F))
        # 지역 인덱스 저장
        self.x_reg = torch.LongTensor(data_df['region_idx'].values)

        # 정답(y) 가격 데이터 스케일링
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = torch.FloatTensor(scaler.transform(dummy)[:, 0].reshape(-1, 1))
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x_seq[idx], self.x_reg[idx], self.y[idx]

# 배치 사이즈 256씩 묶어서 로드 (shuffle=True는 학습 데이터에만 적용)
train_loader = DataLoader(MultiModalDataset(train_df, scaler, len(FEATURES)), batch_size=256, shuffle=True)
val_loader = DataLoader(MultiModalDataset(val_df, scaler, len(FEATURES)), batch_size=256)
test_loader = DataLoader(MultiModalDataset(test_df, scaler, len(FEATURES)), batch_size=256)

# 5. 모델 정의
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=50):
        super().__init__()
        pe = torch.zeros(max_len, d_model) # 일단 50행(시간) × 128열(특징) 크기의 0으로 채워진 빈 지도를 만듦
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1) # 0, 1, 2, ..., 49라는 숫자를 만들고(arange), 이를 세로 기둥 모양(unsqueeze(1))으로 세움. 이것이 각 데이터의 순번이 됨
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)) # 위치 인코딩(Positional Encoding) 공식의 구현체
        pe[:, 0::2] = torch.sin(position * div_term) # 짝수 열
        pe[:, 1::2] = torch.cos(position * div_term) # 홀수 열
        self.register_buffer('pe', pe.unsqueeze(0)) # 가중치가 아니니까 업데이트하지 말고, 모델 저장할 때 같이 저장만
    def forward(self, x): return x + self.pe[:, :x.size(1)]

class MultiRegionTransformer(nn.Module):
    def __init__(self, n_regions, n_features, d_model=128, nhead=8, num_layers=2, emb_dim=16):
        super().__init__()
        self.region_emb = nn.Embedding(n_regions, emb_dim) # 숫자로 된 지역 번호를 16차원의 밀집된 벡터로 변환
        self.feature_emb = nn.Linear(n_features, d_model) # 6개의 아파트 관련 특징(가격, 면적 등)을 128차원의 고차원으로 확장
        self.pos_encoder = PositionalEncoding(d_model) # 사인/코사인 파동을 더해 데이터에 시간 순서
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=256, dropout=0.1, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Sequential(nn.Linear(d_model + emb_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x_s, x_r):
        x = self.pos_encoder(self.feature_emb(x_s)) # 아파트 특징 데이터
        x = self.transformer_encoder(x)
        combined = torch.cat([x[:, -1, :], self.region_emb(x_r)], dim=1) # 앞선 1월부터 11월까지의 변화 양상을 모두 참고하여 업데이트된 현재 시장의 최종적인 특징과 지역 정보 결합
        return self.fc(combined)

model = MultiRegionTransformer(len(region_le.classes_), len(FEATURES)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001); criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 6. 학습
best_val_loss = float('inf'); early_stop_cnt = 0
print(f"{'='*30}\nSTART: Transformer (SAMPLE-WISE SPLIT)\n{'='*30}")

for epoch in range(1, 101):
    model.train(); t_loss = 0
    for xs, xr, y in train_loader:
        xs, xr, y = xs.to(device), xr.to(device), y.to(device)
        optimizer.zero_grad(); loss = criterion(model(xs, xr), y); loss.backward(); optimizer.step(); t_loss += loss.item()
    
    model.eval(); v_loss = 0; all_v_out, all_v_y = [], []
    with torch.no_grad():
        for vx, vr, vy in val_loader:
            vx, vr, vy = vx.to(device), vr.to(device), vy.to(device)
            v_out = model(vx, vr); v_loss += criterion(v_out, vy).item()
            all_v_out.append(v_out.cpu()); all_v_y.append(vy.cpu())
    
    avg_v_loss = v_loss / len(val_loader); scheduler.step(avg_v_loss)
    if epoch % 10 == 0:
        v_p = get_inverse_price(torch.cat(all_v_out).numpy(), scaler, len(FEATURES))
        v_a = get_inverse_price(torch.cat(all_v_y).numpy(), scaler, len(FEATURES))
        print(f"Epoch {epoch:>3} | Loss(T/V): {t_loss/len(train_loader):.4f}/{avg_v_loss:.4f} | {calculate_metrics(v_a, v_p)}")

    if avg_v_loss < best_val_loss:
        best_val_loss = avg_v_loss; torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best_transformer_sample.pth')); early_stop_cnt = 0
    else: early_stop_cnt += 1
    if early_stop_cnt >= 20: break

# 7. 최종 결과
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_transformer_sample.pth'))); model.eval(); all_p, all_a = [], []
with torch.no_grad():
    for tx, tr, ty in test_loader:
        all_p.append(model(tx.to(device), tr.to(device)).cpu()); all_a.append(ty.cpu())
y_p = get_inverse_price(torch.cat(all_p).numpy(), scaler, len(FEATURES))
y_a = get_inverse_price(torch.cat(all_a).numpy(), scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: Transformer (SAMPLE-WISE SPLIT)
Epoch  10 | Loss(T/V): 0.0170/0.0768 | R2: 0.9645 | MAE: 6,115 | RMSE: 10,710 | MAPE: 5.50% | MdAPE: 4.33% | RMSLE: 0.0734
Epoch  20 | Loss(T/V): 0.0144/0.0362 | R2: 0.9834 | MAE: 4,792 | RMSE: 7,325 | MAPE: 5.13% | MdAPE: 4.04% | RMSLE: 0.0704
Epoch  30 | Loss(T/V): 0.0117/0.0347 | R2: 0.9841 | MAE: 4,403 | RMSE: 7,180 | MAPE: 4.31% | MdAPE: 3.20% | RMSLE: 0.0591
Epoch  40 | Loss(T/V): 0.0112/0.0375 | R2: 0.9828 | MAE: 4,687 | RMSE: 7,469 | MAPE: 4.63% | MdAPE: 3.57% | RMSLE: 0.0631
Epoch  50 | Loss(T/V): 0.0106/0.0360 | R2: 0.9835 | MAE: 4,466 | RMSE: 7,317 | MAPE: 4.35% | MdAPE: 3.22% | RMSLE: 0.0596
------------------------------
FINAL TEST RESULT: R2: 0.9271 | MAE: 10,850 | RMSE: 17,392 | MAPE: 9.39% | MdAPE: 6.30% | RMSLE: 0.1242


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings
import math

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리
def load_and_prepare_2():
    data_path = os.path.join(DATA_DIR, 'sido_서울특별시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']

    # [핵심] 동일한 아파트(sample_id) 내에서 비어있는 값을 선형(직선)으로 연결하여 채움
    # limit_direction='both': 앞뒤에 데이터가 하나라도 있으면 양방향으로 확장해서 채움
    for col in FEATURES:
        df[col] = df.groupby('sample_id')[col].transform(lambda x: x.interpolate(method='linear', limit_direction='both'))
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_2()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue # 데이터가 12개 이하인 단지는 학습 불가하므로 제외
    
    sample_windows = []
    values = group[FEATURES].values # 가격, 면적 등 수치 데이터
    reg_idxs = group['region_idx'].values # 해당 단지의 지역 번호
    
    # 슬라이딩 윈도우: 1~12월로 13월 예측, 2~13월로 14월 예측...
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], # 입력: 12개월치 시계열 특징
            'region_idx': reg_idxs[i+WINDOW], # 입력: 지역 정보 (Embedding용)
            'y': values[i+WINDOW, 0] # 정답: 다음 달의 가격(price)
        })
    
    # [데이터 분할] 한 단지의 전체 기록 중 앞 70%는 학습, 10%는 검증, 20%는 테스트용
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
n = len(train_df) + len(val_df) + len(test_df)
if n == 0:
    print("경고: 윈도우가 생성되지 않았습니다.")

# Train 데이터의 평균과 표준편차를 기준으로 전체 데이터 스케일링 (표준화)
scaler = StandardScaler()
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class MultiModalDataset(Dataset):
    def __init__(self, data_df, scaler, n_features):
        # 3차원 배열 생성: (샘플 수, 12개월, 특징 수)
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        # 입력 시퀀스 데이터 스케일링 및 텐서 변환
        self.x_seq = torch.FloatTensor(scaler.transform(x_raw.reshape(-1, F)).reshape(N, W, F))
        # 지역 인덱스 저장
        self.x_reg = torch.LongTensor(data_df['region_idx'].values)

        # 정답(y) 가격 데이터 스케일링
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = torch.FloatTensor(scaler.transform(dummy)[:, 0].reshape(-1, 1))
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x_seq[idx], self.x_reg[idx], self.y[idx]

# 배치 사이즈 256씩 묶어서 로드 (shuffle=True는 학습 데이터에만 적용)
train_loader = DataLoader(MultiModalDataset(train_df, scaler, len(FEATURES)), batch_size=256, shuffle=True)
val_loader = DataLoader(MultiModalDataset(val_df, scaler, len(FEATURES)), batch_size=256)
test_loader = DataLoader(MultiModalDataset(test_df, scaler, len(FEATURES)), batch_size=256)

# 5. 모델 정의
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=50):
        super().__init__()
        pe = torch.zeros(max_len, d_model) # 일단 50행(시간) × 128열(특징) 크기의 0으로 채워진 빈 지도를 만듦
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1) # 0, 1, 2, ..., 49라는 숫자를 만들고(arange), 이를 세로 기둥 모양(unsqueeze(1))으로 세움. 이것이 각 데이터의 순번이 됨
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)) # 위치 인코딩(Positional Encoding) 공식의 구현체
        pe[:, 0::2] = torch.sin(position * div_term) # 짝수 열
        pe[:, 1::2] = torch.cos(position * div_term) # 홀수 열
        self.register_buffer('pe', pe.unsqueeze(0)) # 가중치가 아니니까 업데이트하지 말고, 모델 저장할 때 같이 저장만
    def forward(self, x): return x + self.pe[:, :x.size(1)]

class MultiRegionTransformer(nn.Module):
    def __init__(self, n_regions, n_features, d_model=128, nhead=8, num_layers=2, emb_dim=16):
        super().__init__()
        self.region_emb = nn.Embedding(n_regions, emb_dim) # 숫자로 된 지역 번호를 16차원의 밀집된 벡터로 변환
        self.feature_emb = nn.Linear(n_features, d_model) # 6개의 아파트 관련 특징(가격, 면적 등)을 128차원의 고차원으로 확장
        self.pos_encoder = PositionalEncoding(d_model) # 사인/코사인 파동을 더해 데이터에 시간 순서
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=256, dropout=0.1, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Sequential(nn.Linear(d_model + emb_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x_s, x_r):
        x = self.pos_encoder(self.feature_emb(x_s)) # 아파트 특징 데이터
        x = self.transformer_encoder(x)
        combined = torch.cat([x[:, -1, :], self.region_emb(x_r)], dim=1) # 앞선 1월부터 11월까지의 변화 양상을 모두 참고하여 업데이트된 현재 시장의 최종적인 특징과 지역 정보 결합
        return self.fc(combined)

model = MultiRegionTransformer(len(region_le.classes_), len(FEATURES)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001); criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 6. 학습
best_val_loss = float('inf'); early_stop_cnt = 0
print(f"{'='*30}\nSTART: Transformer (SAMPLE-WISE SPLIT)\n{'='*30}")

for epoch in range(1, 101):
    model.train(); t_loss = 0
    for xs, xr, y in train_loader:
        xs, xr, y = xs.to(device), xr.to(device), y.to(device)
        optimizer.zero_grad(); loss = criterion(model(xs, xr), y); loss.backward(); optimizer.step(); t_loss += loss.item()
    
    model.eval(); v_loss = 0; all_v_out, all_v_y = [], []
    with torch.no_grad():
        for vx, vr, vy in val_loader:
            vx, vr, vy = vx.to(device), vr.to(device), vy.to(device)
            v_out = model(vx, vr); v_loss += criterion(v_out, vy).item()
            all_v_out.append(v_out.cpu()); all_v_y.append(vy.cpu())
    
    avg_v_loss = v_loss / len(val_loader); scheduler.step(avg_v_loss)
    if epoch % 10 == 0:
        v_p = get_inverse_price(torch.cat(all_v_out).numpy(), scaler, len(FEATURES))
        v_a = get_inverse_price(torch.cat(all_v_y).numpy(), scaler, len(FEATURES))
        print(f"Epoch {epoch:>3} | Loss(T/V): {t_loss/len(train_loader):.4f}/{avg_v_loss:.4f} | {calculate_metrics(v_a, v_p)}")

    if avg_v_loss < best_val_loss:
        best_val_loss = avg_v_loss; torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best_transformer_sample.pth')); early_stop_cnt = 0
    else: early_stop_cnt += 1
    if early_stop_cnt >= 20: break

# 7. 최종 결과
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_transformer_sample.pth'))); model.eval(); all_p, all_a = [], []
with torch.no_grad():
    for tx, tr, ty in test_loader:
        all_p.append(model(tx.to(device), tr.to(device)).cpu()); all_a.append(ty.cpu())
y_p = get_inverse_price(torch.cat(all_p).numpy(), scaler, len(FEATURES))
y_a = get_inverse_price(torch.cat(all_a).numpy(), scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: Transformer (SAMPLE-WISE SPLIT)
Epoch  10 | Loss(T/V): 0.0082/0.0299 | R2: 0.9857 | MAE: 5,086 | RMSE: 7,736 | MAPE: 3.98% | MdAPE: 3.30% | RMSLE: 0.0517
Epoch  20 | Loss(T/V): 0.0066/0.0169 | R2: 0.9919 | MAE: 3,255 | RMSE: 5,806 | MAPE: 2.69% | MdAPE: 1.86% | RMSLE: 0.0411
Epoch  30 | Loss(T/V): 0.0061/0.0171 | R2: 0.9918 | MAE: 3,376 | RMSE: 5,838 | MAPE: 2.89% | MdAPE: 2.08% | RMSLE: 0.0425
Epoch  40 | Loss(T/V): 0.0058/0.0182 | R2: 0.9913 | MAE: 3,534 | RMSE: 6,029 | MAPE: 3.07% | MdAPE: 2.18% | RMSLE: 0.0444
------------------------------
FINAL TEST RESULT: R2: 0.9705 | MAE: 8,593 | RMSE: 11,446 | MAPE: 8.18% | MdAPE: 7.06% | RMSLE: 0.0931


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings
import math

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 고도화된 데이터 전처리 (하이브리드: 선형 보간 + 지역 변동률 반영)
def load_and_prepare_advanced():
    data_path = os.path.join(DATA_DIR, 'sido_서울특별시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    
    # [단계 1] 지역별 시장 흐름 파악
    # 같은 구(region_id)의 평균 가격 흐름을 계산
    rt = df.groupby(['region_id', 'transaction_date'])['price'].mean().reset_index().sort_values(['region_id', 'transaction_date'])
    # 3개월 이동평균으로 노이즈 제거 (smooth_price)
    rt['smooth_price'] = rt.groupby('region_id')['price'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
    # 전월 대비 가격이 몇 % 변했는지 계산 (region_multiplier)
    rt['region_multiplier'] = 1 + rt.groupby('region_id')['smooth_price'].pct_change().fillna(0)

    # 원본 데이터에 지역 변동률(multiplier) 결합
    df = pd.merge(df, rt[['region_id', 'transaction_date', 'region_multiplier']], on=['region_id', 'transaction_date'], how='left')
    
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    def apply_advanced_hybrid(group):
        group = group.sort_values('transaction_date')
        p_orig, mult = group['price'].values, group['region_multiplier'].values
        # 방법 A: 단순 선형 보간 값
        p_lin = group['price'].interpolate(method='linear', limit_direction='both').values

        # 실제 거래 데이터가 얼마나 있는지 비율 계산 (충실도)
        actual_count = group['price'].notnull().sum()
        fidelity = actual_count / len(group)

        # [단계 2] 하이브리드 가중치 결정
        # 실제 데이터가 많을수록 선형 보간에 힘을 실어주고(0.8~0.95), 적으면 지역 추세 반영 비중을 높임
        lin_weight = 0.8 + (0.15 * fidelity); reg_weight = 1.0 - lin_weight

        # 방법 B: 이전 실거래가에 지역 변동률을 곱해 나가는 방식 (p_reg)
        p_reg = p_orig.copy()
        for i in range(1, len(p_reg)):
            if np.isnan(p_reg[i]) and not np.isnan(p_reg[i-1]):
                p_reg[i] = p_reg[i-1] * (mult[i] if mult[i] != 0 else 1)
        
        # p_reg의 남은 결측치는 다시 선형 보간으로 보완
        p_reg = pd.Series(p_reg).fillna(pd.Series(p_lin)).values

        # [단계 3] 최종 결합: 원본 데이터는 유지하고, 결측치만 두 방법의 가중치 합으로 채움
        group['price'] = np.where(np.isnan(p_orig), (lin_weight * p_lin) + (reg_weight * p_reg), p_orig)

        # 가격 외 나머지 특징은 단순 선형 보간 처리
        for col in FEATURES[1:]: group[col] = group[col].interpolate(method='linear', limit_direction='both')
        return group
        
    df = df.groupby('sample_id').apply(apply_advanced_hybrid).reset_index(drop=True)
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_advanced()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue # 데이터가 12개 이하인 단지는 학습 불가하므로 제외
    
    sample_windows = []
    values = group[FEATURES].values # 가격, 면적 등 수치 데이터
    reg_idxs = group['region_idx'].values # 해당 단지의 지역 번호
    
    # 슬라이딩 윈도우: 1~12월로 13월 예측, 2~13월로 14월 예측...
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], # 입력: 12개월치 시계열 특징
            'region_idx': reg_idxs[i+WINDOW], # 입력: 지역 정보 (Embedding용)
            'y': values[i+WINDOW, 0] # 정답: 다음 달의 가격(price)
        })
    
    # [데이터 분할] 한 단지의 전체 기록 중 앞 70%는 학습, 10%는 검증, 20%는 테스트용
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
n = len(train_df) + len(val_df) + len(test_df)
if n == 0:
    print("경고: 윈도우가 생성되지 않았습니다.")

# Train 데이터의 평균과 표준편차를 기준으로 전체 데이터 스케일링 (표준화)
scaler = StandardScaler()
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class MultiModalDataset(Dataset):
    def __init__(self, data_df, scaler, n_features):
        # 3차원 배열 생성: (샘플 수, 12개월, 특징 수)
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        # 입력 시퀀스 데이터 스케일링 및 텐서 변환
        self.x_seq = torch.FloatTensor(scaler.transform(x_raw.reshape(-1, F)).reshape(N, W, F))
        # 지역 인덱스 저장
        self.x_reg = torch.LongTensor(data_df['region_idx'].values)

        # 정답(y) 가격 데이터 스케일링
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = torch.FloatTensor(scaler.transform(dummy)[:, 0].reshape(-1, 1))
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x_seq[idx], self.x_reg[idx], self.y[idx]

# 배치 사이즈 256씩 묶어서 로드 (shuffle=True는 학습 데이터에만 적용)
train_loader = DataLoader(MultiModalDataset(train_df, scaler, len(FEATURES)), batch_size=256, shuffle=True)
val_loader = DataLoader(MultiModalDataset(val_df, scaler, len(FEATURES)), batch_size=256)
test_loader = DataLoader(MultiModalDataset(test_df, scaler, len(FEATURES)), batch_size=256)

# 5. 모델 정의
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=50):
        super().__init__()
        pe = torch.zeros(max_len, d_model) # 일단 50행(시간) × 128열(특징) 크기의 0으로 채워진 빈 지도를 만듦
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1) # 0, 1, 2, ..., 49라는 숫자를 만들고(arange), 이를 세로 기둥 모양(unsqueeze(1))으로 세움. 이것이 각 데이터의 순번이 됨
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)) # 위치 인코딩(Positional Encoding) 공식의 구현체
        pe[:, 0::2] = torch.sin(position * div_term) # 짝수 열
        pe[:, 1::2] = torch.cos(position * div_term) # 홀수 열
        self.register_buffer('pe', pe.unsqueeze(0)) # 가중치가 아니니까 업데이트하지 말고, 모델 저장할 때 같이 저장만
    def forward(self, x): return x + self.pe[:, :x.size(1)]

class MultiRegionTransformer(nn.Module):
    def __init__(self, n_regions, n_features, d_model=128, nhead=8, num_layers=2, emb_dim=16):
        super().__init__()
        self.region_emb = nn.Embedding(n_regions, emb_dim) # 숫자로 된 지역 번호를 16차원의 밀집된 벡터로 변환
        self.feature_emb = nn.Linear(n_features, d_model) # 6개의 아파트 관련 특징(가격, 면적 등)을 128차원의 고차원으로 확장
        self.pos_encoder = PositionalEncoding(d_model) # 사인/코사인 파동을 더해 데이터에 시간 순서
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=256, dropout=0.1, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Sequential(nn.Linear(d_model + emb_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x_s, x_r):
        x = self.pos_encoder(self.feature_emb(x_s)) # 아파트 특징 데이터
        x = self.transformer_encoder(x)
        combined = torch.cat([x[:, -1, :], self.region_emb(x_r)], dim=1) # 앞선 1월부터 11월까지의 변화 양상을 모두 참고하여 업데이트된 현재 시장의 최종적인 특징과 지역 정보 결합
        return self.fc(combined)

model = MultiRegionTransformer(len(region_le.classes_), len(FEATURES)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001); criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 6. 학습
best_val_loss = float('inf'); early_stop_cnt = 0
print(f"{'='*30}\nSTART: Transformer (SAMPLE-WISE SPLIT)\n{'='*30}")

for epoch in range(1, 101):
    model.train(); t_loss = 0
    for xs, xr, y in train_loader:
        xs, xr, y = xs.to(device), xr.to(device), y.to(device)
        optimizer.zero_grad(); loss = criterion(model(xs, xr), y); loss.backward(); optimizer.step(); t_loss += loss.item()
    
    model.eval(); v_loss = 0; all_v_out, all_v_y = [], []
    with torch.no_grad():
        for vx, vr, vy in val_loader:
            vx, vr, vy = vx.to(device), vr.to(device), vy.to(device)
            v_out = model(vx, vr); v_loss += criterion(v_out, vy).item()
            all_v_out.append(v_out.cpu()); all_v_y.append(vy.cpu())
    
    avg_v_loss = v_loss / len(val_loader); scheduler.step(avg_v_loss)
    if epoch % 10 == 0:
        v_p = get_inverse_price(torch.cat(all_v_out).numpy(), scaler, len(FEATURES))
        v_a = get_inverse_price(torch.cat(all_v_y).numpy(), scaler, len(FEATURES))
        print(f"Epoch {epoch:>3} | Loss(T/V): {t_loss/len(train_loader):.4f}/{avg_v_loss:.4f} | {calculate_metrics(v_a, v_p)}")

    if avg_v_loss < best_val_loss:
        best_val_loss = avg_v_loss; torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best_transformer_sample.pth')); early_stop_cnt = 0
    else: early_stop_cnt += 1
    if early_stop_cnt >= 20: break

# 7. 최종 결과
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_transformer_sample.pth'))); model.eval(); all_p, all_a = [], []
with torch.no_grad():
    for tx, tr, ty in test_loader:
        all_p.append(model(tx.to(device), tr.to(device)).cpu()); all_a.append(ty.cpu())
y_p = get_inverse_price(torch.cat(all_p).numpy(), scaler, len(FEATURES))
y_a = get_inverse_price(torch.cat(all_a).numpy(), scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: Transformer (SAMPLE-WISE SPLIT)
Epoch  10 | Loss(T/V): 0.0083/0.0248 | R2: 0.9880 | MAE: 4,313 | RMSE: 7,053 | MAPE: 3.47% | MdAPE: 2.74% | RMSLE: 0.0476
Epoch  20 | Loss(T/V): 0.0070/0.0274 | R2: 0.9867 | MAE: 4,325 | RMSE: 7,426 | MAPE: 3.34% | MdAPE: 2.40% | RMSLE: 0.0488
Epoch  30 | Loss(T/V): 0.0060/0.0183 | R2: 0.9912 | MAE: 3,344 | RMSE: 6,053 | MAPE: 2.75% | MdAPE: 1.96% | RMSLE: 0.0416
------------------------------
FINAL TEST RESULT: R2: 0.9734 | MAE: 7,616 | RMSE: 10,870 | MAPE: 6.89% | MdAPE: 5.90% | RMSLE: 0.0802


### 부산

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings
import math

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리 (결측치 행 삭제)
def load_and_prepare():
    data_path = os.path.join(DATA_DIR, 'sido_부산광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    # 결측치가 있는 행 삭제 (실거래 데이터만 남김)
    df = df.dropna(subset=FEATURES)
    
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
n = len(train_df) + len(val_df) + len(test_df)
if n == 0:
    print("경고: 윈도우가 생성되지 않았습니다.")

scaler = StandardScaler()
# Train 데이터 기반으로 스케일러 학습
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class MultiModalDataset(Dataset):
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        self.x_seq = torch.FloatTensor(scaler.transform(x_raw.reshape(-1, F)).reshape(N, W, F))
        self.x_reg = torch.LongTensor(data_df['region_idx'].values)
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = torch.FloatTensor(scaler.transform(dummy)[:, 0].reshape(-1, 1))
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x_seq[idx], self.x_reg[idx], self.y[idx]

train_loader = DataLoader(MultiModalDataset(train_df, scaler, len(FEATURES)), batch_size=256, shuffle=True)
val_loader = DataLoader(MultiModalDataset(val_df, scaler, len(FEATURES)), batch_size=256)
test_loader = DataLoader(MultiModalDataset(test_df, scaler, len(FEATURES)), batch_size=256)

# 5. 모델 정의
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=50):
        super().__init__()
        pe = torch.zeros(max_len, d_model) # 일단 50행(시간) × 128열(특징) 크기의 0으로 채워진 빈 지도를 만듦
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1) # 0, 1, 2, ..., 49라는 숫자를 만들고(arange), 이를 세로 기둥 모양(unsqueeze(1))으로 세움. 이것이 각 데이터의 순번이 됨
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)) # 위치 인코딩(Positional Encoding) 공식의 구현체
        pe[:, 0::2] = torch.sin(position * div_term) # 짝수 열
        pe[:, 1::2] = torch.cos(position * div_term) # 홀수 열
        self.register_buffer('pe', pe.unsqueeze(0)) # 가중치가 아니니까 업데이트하지 말고, 모델 저장할 때 같이 저장만
    def forward(self, x): return x + self.pe[:, :x.size(1)]

class MultiRegionTransformer(nn.Module):
    def __init__(self, n_regions, n_features, d_model=128, nhead=8, num_layers=2, emb_dim=16):
        super().__init__()
        self.region_emb = nn.Embedding(n_regions, emb_dim) # 숫자로 된 지역 번호를 16차원의 밀집된 벡터로 변환
        self.feature_emb = nn.Linear(n_features, d_model) # 6개의 아파트 관련 특징(가격, 면적 등)을 128차원의 고차원으로 확장
        self.pos_encoder = PositionalEncoding(d_model) # 사인/코사인 파동을 더해 데이터에 시간 순서
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=256, dropout=0.1, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Sequential(nn.Linear(d_model + emb_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x_s, x_r):
        x = self.pos_encoder(self.feature_emb(x_s)) # 아파트 특징 데이터
        x = self.transformer_encoder(x)
        combined = torch.cat([x[:, -1, :], self.region_emb(x_r)], dim=1) # 앞선 1월부터 11월까지의 변화 양상을 모두 참고하여 업데이트된 현재 시장의 최종적인 특징과 지역 정보 결합
        return self.fc(combined)

model = MultiRegionTransformer(len(region_le.classes_), len(FEATURES)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001); criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 6. 학습
best_val_loss = float('inf'); early_stop_cnt = 0
print(f"{'='*30}\nSTART: Transformer (SAMPLE-WISE SPLIT)\n{'='*30}")

for epoch in range(1, 101):
    model.train(); t_loss = 0
    for xs, xr, y in train_loader:
        xs, xr, y = xs.to(device), xr.to(device), y.to(device)
        optimizer.zero_grad(); loss = criterion(model(xs, xr), y); loss.backward(); optimizer.step(); t_loss += loss.item()
    
    model.eval(); v_loss = 0; all_v_out, all_v_y = [], []
    with torch.no_grad():
        for vx, vr, vy in val_loader:
            vx, vr, vy = vx.to(device), vr.to(device), vy.to(device)
            v_out = model(vx, vr); v_loss += criterion(v_out, vy).item()
            all_v_out.append(v_out.cpu()); all_v_y.append(vy.cpu())
    
    avg_v_loss = v_loss / len(val_loader); scheduler.step(avg_v_loss)
    if epoch % 10 == 0:
        v_p = get_inverse_price(torch.cat(all_v_out).numpy(), scaler, len(FEATURES))
        v_a = get_inverse_price(torch.cat(all_v_y).numpy(), scaler, len(FEATURES))
        print(f"Epoch {epoch:>3} | Loss(T/V): {t_loss/len(train_loader):.4f}/{avg_v_loss:.4f} | {calculate_metrics(v_a, v_p)}")

    if avg_v_loss < best_val_loss:
        best_val_loss = avg_v_loss; torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best_transformer_sample.pth')); early_stop_cnt = 0
    else: early_stop_cnt += 1
    if early_stop_cnt >= 20: break

# 7. 최종 결과
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_transformer_sample.pth'))); model.eval(); all_p, all_a = [], []
with torch.no_grad():
    for tx, tr, ty in test_loader:
        all_p.append(model(tx.to(device), tr.to(device)).cpu()); all_a.append(ty.cpu())
y_p = get_inverse_price(torch.cat(all_p).numpy(), scaler, len(FEATURES))
y_a = get_inverse_price(torch.cat(all_a).numpy(), scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: Transformer (SAMPLE-WISE SPLIT)
Epoch  10 | Loss(T/V): 0.0200/0.0489 | R2: 0.9719 | MAE: 1,699 | RMSE: 3,384 | MAPE: 5.46% | MdAPE: 4.05% | RMSLE: 0.0751
Epoch  20 | Loss(T/V): 0.0170/0.0463 | R2: 0.9735 | MAE: 1,908 | RMSE: 3,288 | MAPE: 6.77% | MdAPE: 5.65% | RMSLE: 0.0903
Epoch  30 | Loss(T/V): 0.0153/0.0484 | R2: 0.9722 | MAE: 1,787 | RMSE: 3,368 | MAPE: 5.88% | MdAPE: 4.74% | RMSLE: 0.0786
------------------------------
FINAL TEST RESULT: R2: 0.8857 | MAE: 4,511 | RMSE: 9,043 | MAPE: 11.06% | MdAPE: 8.11% | RMSLE: 0.1529


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings
import math

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리
def load_and_prepare_2():
    data_path = os.path.join(DATA_DIR, 'sido_부산광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    for col in FEATURES:
        df[col] = df.groupby('sample_id')[col].transform(lambda x: x.interpolate(method='linear', limit_direction='both'))
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_2()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
n = len(train_df) + len(val_df) + len(test_df)
if n == 0:
    print("경고: 윈도우가 생성되지 않았습니다.")

scaler = StandardScaler()
# Train 데이터 기반으로 스케일러 학습
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class MultiModalDataset(Dataset):
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        self.x_seq = torch.FloatTensor(scaler.transform(x_raw.reshape(-1, F)).reshape(N, W, F))
        self.x_reg = torch.LongTensor(data_df['region_idx'].values)
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = torch.FloatTensor(scaler.transform(dummy)[:, 0].reshape(-1, 1))
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x_seq[idx], self.x_reg[idx], self.y[idx]

train_loader = DataLoader(MultiModalDataset(train_df, scaler, len(FEATURES)), batch_size=256, shuffle=True)
val_loader = DataLoader(MultiModalDataset(val_df, scaler, len(FEATURES)), batch_size=256)
test_loader = DataLoader(MultiModalDataset(test_df, scaler, len(FEATURES)), batch_size=256)

# 5. 모델 정의
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=50):
        super().__init__()
        pe = torch.zeros(max_len, d_model) # 일단 50행(시간) × 128열(특징) 크기의 0으로 채워진 빈 지도를 만듦
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1) # 0, 1, 2, ..., 49라는 숫자를 만들고(arange), 이를 세로 기둥 모양(unsqueeze(1))으로 세움. 이것이 각 데이터의 순번이 됨
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)) # 위치 인코딩(Positional Encoding) 공식의 구현체
        pe[:, 0::2] = torch.sin(position * div_term) # 짝수 열
        pe[:, 1::2] = torch.cos(position * div_term) # 홀수 열
        self.register_buffer('pe', pe.unsqueeze(0)) # 가중치가 아니니까 업데이트하지 말고, 모델 저장할 때 같이 저장만
    def forward(self, x): return x + self.pe[:, :x.size(1)]

class MultiRegionTransformer(nn.Module):
    def __init__(self, n_regions, n_features, d_model=128, nhead=8, num_layers=2, emb_dim=16):
        super().__init__()
        self.region_emb = nn.Embedding(n_regions, emb_dim) # 숫자로 된 지역 번호를 16차원의 밀집된 벡터로 변환
        self.feature_emb = nn.Linear(n_features, d_model) # 6개의 아파트 관련 특징(가격, 면적 등)을 128차원의 고차원으로 확장
        self.pos_encoder = PositionalEncoding(d_model) # 사인/코사인 파동을 더해 데이터에 시간 순서
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=256, dropout=0.1, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Sequential(nn.Linear(d_model + emb_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x_s, x_r):
        x = self.pos_encoder(self.feature_emb(x_s)) # 아파트 특징 데이터
        x = self.transformer_encoder(x)
        combined = torch.cat([x[:, -1, :], self.region_emb(x_r)], dim=1) # 앞선 1월부터 11월까지의 변화 양상을 모두 참고하여 업데이트된 현재 시장의 최종적인 특징과 지역 정보 결합
        return self.fc(combined)

model = MultiRegionTransformer(len(region_le.classes_), len(FEATURES)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001); criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 6. 학습
best_val_loss = float('inf'); early_stop_cnt = 0
print(f"{'='*30}\nSTART: Transformer (SAMPLE-WISE SPLIT)\n{'='*30}")

for epoch in range(1, 101):
    model.train(); t_loss = 0
    for xs, xr, y in train_loader:
        xs, xr, y = xs.to(device), xr.to(device), y.to(device)
        optimizer.zero_grad(); loss = criterion(model(xs, xr), y); loss.backward(); optimizer.step(); t_loss += loss.item()
    
    model.eval(); v_loss = 0; all_v_out, all_v_y = [], []
    with torch.no_grad():
        for vx, vr, vy in val_loader:
            vx, vr, vy = vx.to(device), vr.to(device), vy.to(device)
            v_out = model(vx, vr); v_loss += criterion(v_out, vy).item()
            all_v_out.append(v_out.cpu()); all_v_y.append(vy.cpu())
    
    avg_v_loss = v_loss / len(val_loader); scheduler.step(avg_v_loss)
    if epoch % 10 == 0:
        v_p = get_inverse_price(torch.cat(all_v_out).numpy(), scaler, len(FEATURES))
        v_a = get_inverse_price(torch.cat(all_v_y).numpy(), scaler, len(FEATURES))
        print(f"Epoch {epoch:>3} | Loss(T/V): {t_loss/len(train_loader):.4f}/{avg_v_loss:.4f} | {calculate_metrics(v_a, v_p)}")

    if avg_v_loss < best_val_loss:
        best_val_loss = avg_v_loss; torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best_transformer_sample.pth')); early_stop_cnt = 0
    else: early_stop_cnt += 1
    if early_stop_cnt >= 20: break

# 7. 최종 결과
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_transformer_sample.pth'))); model.eval(); all_p, all_a = [], []
with torch.no_grad():
    for tx, tr, ty in test_loader:
        all_p.append(model(tx.to(device), tr.to(device)).cpu()); all_a.append(ty.cpu())
y_p = get_inverse_price(torch.cat(all_p).numpy(), scaler, len(FEATURES))
y_a = get_inverse_price(torch.cat(all_a).numpy(), scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: Transformer (SAMPLE-WISE SPLIT)
Epoch  10 | Loss(T/V): 0.0184/0.0438 | R2: 0.9868 | MAE: 1,663 | RMSE: 3,476 | MAPE: 4.39% | MdAPE: 3.34% | RMSLE: 0.0631
Epoch  20 | Loss(T/V): 0.0131/0.0387 | R2: 0.9888 | MAE: 1,849 | RMSE: 3,194 | MAPE: 4.37% | MdAPE: 3.86% | RMSLE: 0.0544
Epoch  30 | Loss(T/V): 0.0120/0.0458 | R2: 0.9869 | MAE: 2,002 | RMSE: 3,468 | MAPE: 4.49% | MdAPE: 3.94% | RMSLE: 0.0545
------------------------------
FINAL TEST RESULT: R2: 0.9611 | MAE: 3,762 | RMSE: 6,045 | MAPE: 10.12% | MdAPE: 8.14% | RMSLE: 0.1176


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings
import math

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 고도화된 데이터 전처리 (하이브리드)
def load_and_prepare_advanced():
    data_path = os.path.join(DATA_DIR, 'sido_부산광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    
    rt = df.groupby(['region_id', 'transaction_date'])['price'].mean().reset_index().sort_values(['region_id', 'transaction_date'])
    rt['smooth_price'] = rt.groupby('region_id')['price'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
    rt['region_multiplier'] = 1 + rt.groupby('region_id')['smooth_price'].pct_change().fillna(0)
    df = pd.merge(df, rt[['region_id', 'transaction_date', 'region_multiplier']], on=['region_id', 'transaction_date'], how='left')
    
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    def apply_advanced_hybrid(group):
        group = group.sort_values('transaction_date')
        p_orig, mult = group['price'].values, group['region_multiplier'].values
        p_lin = group['price'].interpolate(method='linear', limit_direction='both').values
        actual_count = group['price'].notnull().sum()
        fidelity = actual_count / len(group)
        lin_weight = 0.8 + (0.15 * fidelity); reg_weight = 1.0 - lin_weight
        p_reg = p_orig.copy()
        for i in range(1, len(p_reg)):
            if np.isnan(p_reg[i]) and not np.isnan(p_reg[i-1]):
                p_reg[i] = p_reg[i-1] * (mult[i] if mult[i] != 0 else 1)
        p_reg = pd.Series(p_reg).fillna(pd.Series(p_lin)).values
        group['price'] = np.where(np.isnan(p_orig), (lin_weight * p_lin) + (reg_weight * p_reg), p_orig)
        for col in FEATURES[1:]: group[col] = group[col].interpolate(method='linear', limit_direction='both')
        return group
        
    df = df.groupby('sample_id').apply(apply_advanced_hybrid).reset_index(drop=True)
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_advanced()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
n = len(train_df) + len(val_df) + len(test_df)
if n == 0:
    print("경고: 윈도우가 생성되지 않았습니다.")

scaler = StandardScaler()
# Train 데이터 기반으로 스케일러 학습
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class MultiModalDataset(Dataset):
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        self.x_seq = torch.FloatTensor(scaler.transform(x_raw.reshape(-1, F)).reshape(N, W, F))
        self.x_reg = torch.LongTensor(data_df['region_idx'].values)
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = torch.FloatTensor(scaler.transform(dummy)[:, 0].reshape(-1, 1))
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x_seq[idx], self.x_reg[idx], self.y[idx]

train_loader = DataLoader(MultiModalDataset(train_df, scaler, len(FEATURES)), batch_size=256, shuffle=True)
val_loader = DataLoader(MultiModalDataset(val_df, scaler, len(FEATURES)), batch_size=256)
test_loader = DataLoader(MultiModalDataset(test_df, scaler, len(FEATURES)), batch_size=256)

# 5. 모델 정의
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=50):
        super().__init__()
        pe = torch.zeros(max_len, d_model) # 일단 50행(시간) × 128열(특징) 크기의 0으로 채워진 빈 지도를 만듦
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1) # 0, 1, 2, ..., 49라는 숫자를 만들고(arange), 이를 세로 기둥 모양(unsqueeze(1))으로 세움. 이것이 각 데이터의 순번이 됨
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)) # 위치 인코딩(Positional Encoding) 공식의 구현체
        pe[:, 0::2] = torch.sin(position * div_term) # 짝수 열
        pe[:, 1::2] = torch.cos(position * div_term) # 홀수 열
        self.register_buffer('pe', pe.unsqueeze(0)) # 가중치가 아니니까 업데이트하지 말고, 모델 저장할 때 같이 저장만
    def forward(self, x): return x + self.pe[:, :x.size(1)]

class MultiRegionTransformer(nn.Module):
    def __init__(self, n_regions, n_features, d_model=128, nhead=8, num_layers=2, emb_dim=16):
        super().__init__()
        self.region_emb = nn.Embedding(n_regions, emb_dim) # 숫자로 된 지역 번호를 16차원의 밀집된 벡터로 변환
        self.feature_emb = nn.Linear(n_features, d_model) # 6개의 아파트 관련 특징(가격, 면적 등)을 128차원의 고차원으로 확장
        self.pos_encoder = PositionalEncoding(d_model) # 사인/코사인 파동을 더해 데이터에 시간 순서
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=256, dropout=0.1, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Sequential(nn.Linear(d_model + emb_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x_s, x_r):
        x = self.pos_encoder(self.feature_emb(x_s)) # 아파트 특징 데이터
        x = self.transformer_encoder(x)
        combined = torch.cat([x[:, -1, :], self.region_emb(x_r)], dim=1) # 앞선 1월부터 11월까지의 변화 양상을 모두 참고하여 업데이트된 현재 시장의 최종적인 특징과 지역 정보 결합
        return self.fc(combined)

model = MultiRegionTransformer(len(region_le.classes_), len(FEATURES)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001); criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 6. 학습
best_val_loss = float('inf'); early_stop_cnt = 0
print(f"{'='*30}\nSTART: Transformer (SAMPLE-WISE SPLIT)\n{'='*30}")

for epoch in range(1, 101):
    model.train(); t_loss = 0
    for xs, xr, y in train_loader:
        xs, xr, y = xs.to(device), xr.to(device), y.to(device)
        optimizer.zero_grad(); loss = criterion(model(xs, xr), y); loss.backward(); optimizer.step(); t_loss += loss.item()
    
    model.eval(); v_loss = 0; all_v_out, all_v_y = [], []
    with torch.no_grad():
        for vx, vr, vy in val_loader:
            vx, vr, vy = vx.to(device), vr.to(device), vy.to(device)
            v_out = model(vx, vr); v_loss += criterion(v_out, vy).item()
            all_v_out.append(v_out.cpu()); all_v_y.append(vy.cpu())
    
    avg_v_loss = v_loss / len(val_loader); scheduler.step(avg_v_loss)
    if epoch % 10 == 0:
        v_p = get_inverse_price(torch.cat(all_v_out).numpy(), scaler, len(FEATURES))
        v_a = get_inverse_price(torch.cat(all_v_y).numpy(), scaler, len(FEATURES))
        print(f"Epoch {epoch:>3} | Loss(T/V): {t_loss/len(train_loader):.4f}/{avg_v_loss:.4f} | {calculate_metrics(v_a, v_p)}")

    if avg_v_loss < best_val_loss:
        best_val_loss = avg_v_loss; torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best_transformer_sample.pth')); early_stop_cnt = 0
    else: early_stop_cnt += 1
    if early_stop_cnt >= 20: break

# 7. 최종 결과
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_transformer_sample.pth'))); model.eval(); all_p, all_a = [], []
with torch.no_grad():
    for tx, tr, ty in test_loader:
        all_p.append(model(tx.to(device), tr.to(device)).cpu()); all_a.append(ty.cpu())
y_p = get_inverse_price(torch.cat(all_p).numpy(), scaler, len(FEATURES))
y_a = get_inverse_price(torch.cat(all_a).numpy(), scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: Transformer (SAMPLE-WISE SPLIT)
Epoch  10 | Loss(T/V): 0.0182/0.0376 | R2: 0.9881 | MAE: 1,708 | RMSE: 3,216 | MAPE: 4.69% | MdAPE: 3.57% | RMSLE: 0.0660
Epoch  20 | Loss(T/V): 0.0131/0.0342 | R2: 0.9896 | MAE: 1,861 | RMSE: 3,005 | MAPE: 4.43% | MdAPE: 3.93% | RMSLE: 0.0549
Epoch  30 | Loss(T/V): 0.0120/0.0417 | R2: 0.9874 | MAE: 1,997 | RMSE: 3,313 | MAPE: 4.44% | MdAPE: 3.89% | RMSLE: 0.0534
------------------------------
FINAL TEST RESULT: R2: 0.9655 | MAE: 3,494 | RMSE: 5,644 | MAPE: 9.22% | MdAPE: 7.38% | RMSLE: 0.1098


### 대구

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings
import math

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리 (결측치 행 삭제)
def load_and_prepare():
    data_path = os.path.join(DATA_DIR, 'sido_대구광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    # 결측치가 있는 행 삭제 (실거래 데이터만 남김)
    df = df.dropna(subset=FEATURES)
    
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
n = len(train_df) + len(val_df) + len(test_df)
if n == 0:
    print("경고: 윈도우가 생성되지 않았습니다.")

scaler = StandardScaler()
# Train 데이터 기반으로 스케일러 학습
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class MultiModalDataset(Dataset):
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        self.x_seq = torch.FloatTensor(scaler.transform(x_raw.reshape(-1, F)).reshape(N, W, F))
        self.x_reg = torch.LongTensor(data_df['region_idx'].values)
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = torch.FloatTensor(scaler.transform(dummy)[:, 0].reshape(-1, 1))
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x_seq[idx], self.x_reg[idx], self.y[idx]

train_loader = DataLoader(MultiModalDataset(train_df, scaler, len(FEATURES)), batch_size=256, shuffle=True)
val_loader = DataLoader(MultiModalDataset(val_df, scaler, len(FEATURES)), batch_size=256)
test_loader = DataLoader(MultiModalDataset(test_df, scaler, len(FEATURES)), batch_size=256)

# 5. 모델 정의
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=50):
        super().__init__()
        pe = torch.zeros(max_len, d_model) # 일단 50행(시간) × 128열(특징) 크기의 0으로 채워진 빈 지도를 만듦
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1) # 0, 1, 2, ..., 49라는 숫자를 만들고(arange), 이를 세로 기둥 모양(unsqueeze(1))으로 세움. 이것이 각 데이터의 순번이 됨
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)) # 위치 인코딩(Positional Encoding) 공식의 구현체
        pe[:, 0::2] = torch.sin(position * div_term) # 짝수 열
        pe[:, 1::2] = torch.cos(position * div_term) # 홀수 열
        self.register_buffer('pe', pe.unsqueeze(0)) # 가중치가 아니니까 업데이트하지 말고, 모델 저장할 때 같이 저장만
    def forward(self, x): return x + self.pe[:, :x.size(1)]

class MultiRegionTransformer(nn.Module):
    def __init__(self, n_regions, n_features, d_model=128, nhead=8, num_layers=2, emb_dim=16):
        super().__init__()
        self.region_emb = nn.Embedding(n_regions, emb_dim) # 숫자로 된 지역 번호를 16차원의 밀집된 벡터로 변환
        self.feature_emb = nn.Linear(n_features, d_model) # 6개의 아파트 관련 특징(가격, 면적 등)을 128차원의 고차원으로 확장
        self.pos_encoder = PositionalEncoding(d_model) # 사인/코사인 파동을 더해 데이터에 시간 순서
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=256, dropout=0.1, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Sequential(nn.Linear(d_model + emb_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x_s, x_r):
        x = self.pos_encoder(self.feature_emb(x_s)) # 아파트 특징 데이터
        x = self.transformer_encoder(x)
        combined = torch.cat([x[:, -1, :], self.region_emb(x_r)], dim=1) # 앞선 1월부터 11월까지의 변화 양상을 모두 참고하여 업데이트된 현재 시장의 최종적인 특징과 지역 정보 결합
        return self.fc(combined)

model = MultiRegionTransformer(len(region_le.classes_), len(FEATURES)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001); criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 6. 학습
best_val_loss = float('inf'); early_stop_cnt = 0
print(f"{'='*30}\nSTART: Transformer (SAMPLE-WISE SPLIT)\n{'='*30}")

for epoch in range(1, 101):
    model.train(); t_loss = 0
    for xs, xr, y in train_loader:
        xs, xr, y = xs.to(device), xr.to(device), y.to(device)
        optimizer.zero_grad(); loss = criterion(model(xs, xr), y); loss.backward(); optimizer.step(); t_loss += loss.item()
    
    model.eval(); v_loss = 0; all_v_out, all_v_y = [], []
    with torch.no_grad():
        for vx, vr, vy in val_loader:
            vx, vr, vy = vx.to(device), vr.to(device), vy.to(device)
            v_out = model(vx, vr); v_loss += criterion(v_out, vy).item()
            all_v_out.append(v_out.cpu()); all_v_y.append(vy.cpu())
    
    avg_v_loss = v_loss / len(val_loader); scheduler.step(avg_v_loss)
    if epoch % 10 == 0:
        v_p = get_inverse_price(torch.cat(all_v_out).numpy(), scaler, len(FEATURES))
        v_a = get_inverse_price(torch.cat(all_v_y).numpy(), scaler, len(FEATURES))
        print(f"Epoch {epoch:>3} | Loss(T/V): {t_loss/len(train_loader):.4f}/{avg_v_loss:.4f} | {calculate_metrics(v_a, v_p)}")

    if avg_v_loss < best_val_loss:
        best_val_loss = avg_v_loss; torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best_transformer_sample.pth')); early_stop_cnt = 0
    else: early_stop_cnt += 1
    if early_stop_cnt >= 20: break

# 7. 최종 결과
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_transformer_sample.pth'))); model.eval(); all_p, all_a = [], []
with torch.no_grad():
    for tx, tr, ty in test_loader:
        all_p.append(model(tx.to(device), tr.to(device)).cpu()); all_a.append(ty.cpu())
y_p = get_inverse_price(torch.cat(all_p).numpy(), scaler, len(FEATURES))
y_a = get_inverse_price(torch.cat(all_a).numpy(), scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: Transformer (SAMPLE-WISE SPLIT)
Epoch  10 | Loss(T/V): 0.0159/0.0240 | R2: 0.9832 | MAE: 1,075 | RMSE: 1,749 | MAPE: 4.73% | MdAPE: 3.42% | RMSLE: 0.0708
Epoch  20 | Loss(T/V): 0.0142/0.0269 | R2: 0.9811 | MAE: 1,143 | RMSE: 1,854 | MAPE: 4.59% | MdAPE: 3.62% | RMSLE: 0.0646
Epoch  30 | Loss(T/V): 0.0126/0.0226 | R2: 0.9842 | MAE: 977 | RMSE: 1,696 | MAPE: 4.04% | MdAPE: 2.90% | RMSLE: 0.0597
Epoch  40 | Loss(T/V): 0.0114/0.0254 | R2: 0.9822 | MAE: 1,081 | RMSE: 1,798 | MAPE: 4.62% | MdAPE: 3.40% | RMSLE: 0.0697
------------------------------
FINAL TEST RESULT: R2: 0.9419 | MAE: 2,086 | RMSE: 3,349 | MAPE: 8.07% | MdAPE: 5.64% | RMSLE: 0.1063


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings
import math

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리
def load_and_prepare_2():
    data_path = os.path.join(DATA_DIR, 'sido_대구광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    for col in FEATURES:
        df[col] = df.groupby('sample_id')[col].transform(lambda x: x.interpolate(method='linear', limit_direction='both'))
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_2()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
n = len(train_df) + len(val_df) + len(test_df)
if n == 0:
    print("경고: 윈도우가 생성되지 않았습니다.")

scaler = StandardScaler()
# Train 데이터 기반으로 스케일러 학습
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class MultiModalDataset(Dataset):
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        self.x_seq = torch.FloatTensor(scaler.transform(x_raw.reshape(-1, F)).reshape(N, W, F))
        self.x_reg = torch.LongTensor(data_df['region_idx'].values)
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = torch.FloatTensor(scaler.transform(dummy)[:, 0].reshape(-1, 1))
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x_seq[idx], self.x_reg[idx], self.y[idx]

train_loader = DataLoader(MultiModalDataset(train_df, scaler, len(FEATURES)), batch_size=256, shuffle=True)
val_loader = DataLoader(MultiModalDataset(val_df, scaler, len(FEATURES)), batch_size=256)
test_loader = DataLoader(MultiModalDataset(test_df, scaler, len(FEATURES)), batch_size=256)

# 5. 모델 정의
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=50):
        super().__init__()
        pe = torch.zeros(max_len, d_model) # 일단 50행(시간) × 128열(특징) 크기의 0으로 채워진 빈 지도를 만듦
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1) # 0, 1, 2, ..., 49라는 숫자를 만들고(arange), 이를 세로 기둥 모양(unsqueeze(1))으로 세움. 이것이 각 데이터의 순번이 됨
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)) # 위치 인코딩(Positional Encoding) 공식의 구현체
        pe[:, 0::2] = torch.sin(position * div_term) # 짝수 열
        pe[:, 1::2] = torch.cos(position * div_term) # 홀수 열
        self.register_buffer('pe', pe.unsqueeze(0)) # 가중치가 아니니까 업데이트하지 말고, 모델 저장할 때 같이 저장만
    def forward(self, x): return x + self.pe[:, :x.size(1)]

class MultiRegionTransformer(nn.Module):
    def __init__(self, n_regions, n_features, d_model=128, nhead=8, num_layers=2, emb_dim=16):
        super().__init__()
        self.region_emb = nn.Embedding(n_regions, emb_dim) # 숫자로 된 지역 번호를 16차원의 밀집된 벡터로 변환
        self.feature_emb = nn.Linear(n_features, d_model) # 6개의 아파트 관련 특징(가격, 면적 등)을 128차원의 고차원으로 확장
        self.pos_encoder = PositionalEncoding(d_model) # 사인/코사인 파동을 더해 데이터에 시간 순서
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=256, dropout=0.1, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Sequential(nn.Linear(d_model + emb_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x_s, x_r):
        x = self.pos_encoder(self.feature_emb(x_s)) # 아파트 특징 데이터
        x = self.transformer_encoder(x)
        combined = torch.cat([x[:, -1, :], self.region_emb(x_r)], dim=1) # 앞선 1월부터 11월까지의 변화 양상을 모두 참고하여 업데이트된 현재 시장의 최종적인 특징과 지역 정보 결합
        return self.fc(combined)

model = MultiRegionTransformer(len(region_le.classes_), len(FEATURES)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001); criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 6. 학습
best_val_loss = float('inf'); early_stop_cnt = 0
print(f"{'='*30}\nSTART: Transformer (SAMPLE-WISE SPLIT)\n{'='*30}")

for epoch in range(1, 101):
    model.train(); t_loss = 0
    for xs, xr, y in train_loader:
        xs, xr, y = xs.to(device), xr.to(device), y.to(device)
        optimizer.zero_grad(); loss = criterion(model(xs, xr), y); loss.backward(); optimizer.step(); t_loss += loss.item()
    
    model.eval(); v_loss = 0; all_v_out, all_v_y = [], []
    with torch.no_grad():
        for vx, vr, vy in val_loader:
            vx, vr, vy = vx.to(device), vr.to(device), vy.to(device)
            v_out = model(vx, vr); v_loss += criterion(v_out, vy).item()
            all_v_out.append(v_out.cpu()); all_v_y.append(vy.cpu())
    
    avg_v_loss = v_loss / len(val_loader); scheduler.step(avg_v_loss)
    if epoch % 10 == 0:
        v_p = get_inverse_price(torch.cat(all_v_out).numpy(), scaler, len(FEATURES))
        v_a = get_inverse_price(torch.cat(all_v_y).numpy(), scaler, len(FEATURES))
        print(f"Epoch {epoch:>3} | Loss(T/V): {t_loss/len(train_loader):.4f}/{avg_v_loss:.4f} | {calculate_metrics(v_a, v_p)}")

    if avg_v_loss < best_val_loss:
        best_val_loss = avg_v_loss; torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best_transformer_sample.pth')); early_stop_cnt = 0
    else: early_stop_cnt += 1
    if early_stop_cnt >= 20: break

# 7. 최종 결과
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_transformer_sample.pth'))); model.eval(); all_p, all_a = [], []
with torch.no_grad():
    for tx, tr, ty in test_loader:
        all_p.append(model(tx.to(device), tr.to(device)).cpu()); all_a.append(ty.cpu())
y_p = get_inverse_price(torch.cat(all_p).numpy(), scaler, len(FEATURES))
y_a = get_inverse_price(torch.cat(all_a).numpy(), scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: Transformer (SAMPLE-WISE SPLIT)
Epoch  10 | Loss(T/V): 0.0128/0.0209 | R2: 0.9878 | MAE: 1,097 | RMSE: 1,783 | MAPE: 4.62% | MdAPE: 3.31% | RMSLE: 0.0634
Epoch  20 | Loss(T/V): 0.0104/0.0104 | R2: 0.9940 | MAE: 768 | RMSE: 1,254 | MAPE: 2.66% | MdAPE: 2.16% | RMSLE: 0.0341
Epoch  30 | Loss(T/V): 0.0100/0.0083 | R2: 0.9953 | MAE: 732 | RMSE: 1,102 | MAPE: 2.54% | MdAPE: 2.15% | RMSLE: 0.0312
Epoch  40 | Loss(T/V): 0.0085/0.0126 | R2: 0.9928 | MAE: 691 | RMSE: 1,367 | MAPE: 2.29% | MdAPE: 1.76% | RMSLE: 0.0325
------------------------------
FINAL TEST RESULT: R2: 0.9751 | MAE: 1,701 | RMSE: 2,422 | MAPE: 6.86% | MdAPE: 5.33% | RMSLE: 0.0838


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings
import math

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 고도화된 데이터 전처리 (하이브리드)
def load_and_prepare_advanced():
    data_path = os.path.join(DATA_DIR, 'sido_대구광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    
    rt = df.groupby(['region_id', 'transaction_date'])['price'].mean().reset_index().sort_values(['region_id', 'transaction_date'])
    rt['smooth_price'] = rt.groupby('region_id')['price'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
    rt['region_multiplier'] = 1 + rt.groupby('region_id')['smooth_price'].pct_change().fillna(0)
    df = pd.merge(df, rt[['region_id', 'transaction_date', 'region_multiplier']], on=['region_id', 'transaction_date'], how='left')
    
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    def apply_advanced_hybrid(group):
        group = group.sort_values('transaction_date')
        p_orig, mult = group['price'].values, group['region_multiplier'].values
        p_lin = group['price'].interpolate(method='linear', limit_direction='both').values
        actual_count = group['price'].notnull().sum()
        fidelity = actual_count / len(group)
        lin_weight = 0.8 + (0.15 * fidelity); reg_weight = 1.0 - lin_weight
        p_reg = p_orig.copy()
        for i in range(1, len(p_reg)):
            if np.isnan(p_reg[i]) and not np.isnan(p_reg[i-1]):
                p_reg[i] = p_reg[i-1] * (mult[i] if mult[i] != 0 else 1)
        p_reg = pd.Series(p_reg).fillna(pd.Series(p_lin)).values
        group['price'] = np.where(np.isnan(p_orig), (lin_weight * p_lin) + (reg_weight * p_reg), p_orig)
        for col in FEATURES[1:]: group[col] = group[col].interpolate(method='linear', limit_direction='both')
        return group
        
    df = df.groupby('sample_id').apply(apply_advanced_hybrid).reset_index(drop=True)
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_advanced()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    # [사용자 요청 로직 적용]
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
n = len(train_df) + len(val_df) + len(test_df)
if n == 0:
    print("경고: 윈도우가 생성되지 않았습니다.")

scaler = StandardScaler()
# Train 데이터 기반으로 스케일러 학습
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class MultiModalDataset(Dataset):
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        self.x_seq = torch.FloatTensor(scaler.transform(x_raw.reshape(-1, F)).reshape(N, W, F))
        self.x_reg = torch.LongTensor(data_df['region_idx'].values)
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = torch.FloatTensor(scaler.transform(dummy)[:, 0].reshape(-1, 1))
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x_seq[idx], self.x_reg[idx], self.y[idx]

train_loader = DataLoader(MultiModalDataset(train_df, scaler, len(FEATURES)), batch_size=256, shuffle=True)
val_loader = DataLoader(MultiModalDataset(val_df, scaler, len(FEATURES)), batch_size=256)
test_loader = DataLoader(MultiModalDataset(test_df, scaler, len(FEATURES)), batch_size=256)

# 5. 모델 정의
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=50):
        super().__init__()
        pe = torch.zeros(max_len, d_model) # 일단 50행(시간) × 128열(특징) 크기의 0으로 채워진 빈 지도를 만듦
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1) # 0, 1, 2, ..., 49라는 숫자를 만들고(arange), 이를 세로 기둥 모양(unsqueeze(1))으로 세움. 이것이 각 데이터의 순번이 됨
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)) # 위치 인코딩(Positional Encoding) 공식의 구현체
        pe[:, 0::2] = torch.sin(position * div_term) # 짝수 열
        pe[:, 1::2] = torch.cos(position * div_term) # 홀수 열
        self.register_buffer('pe', pe.unsqueeze(0)) # 가중치가 아니니까 업데이트하지 말고, 모델 저장할 때 같이 저장만
    def forward(self, x): return x + self.pe[:, :x.size(1)]

class MultiRegionTransformer(nn.Module):
    def __init__(self, n_regions, n_features, d_model=128, nhead=8, num_layers=2, emb_dim=16):
        super().__init__()
        self.region_emb = nn.Embedding(n_regions, emb_dim) # 숫자로 된 지역 번호를 16차원의 밀집된 벡터로 변환
        self.feature_emb = nn.Linear(n_features, d_model) # 6개의 아파트 관련 특징(가격, 면적 등)을 128차원의 고차원으로 확장
        self.pos_encoder = PositionalEncoding(d_model) # 사인/코사인 파동을 더해 데이터에 시간 순서
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=256, dropout=0.1, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Sequential(nn.Linear(d_model + emb_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x_s, x_r):
        x = self.pos_encoder(self.feature_emb(x_s)) # 아파트 특징 데이터
        x = self.transformer_encoder(x)
        combined = torch.cat([x[:, -1, :], self.region_emb(x_r)], dim=1) # 앞선 1월부터 11월까지의 변화 양상을 모두 참고하여 업데이트된 현재 시장의 최종적인 특징과 지역 정보 결합
        return self.fc(combined)

model = MultiRegionTransformer(len(region_le.classes_), len(FEATURES)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001); criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 6. 학습
best_val_loss = float('inf'); early_stop_cnt = 0
print(f"{'='*30}\nSTART: Transformer (SAMPLE-WISE SPLIT)\n{'='*30}")

for epoch in range(1, 101):
    model.train(); t_loss = 0
    for xs, xr, y in train_loader:
        xs, xr, y = xs.to(device), xr.to(device), y.to(device)
        optimizer.zero_grad(); loss = criterion(model(xs, xr), y); loss.backward(); optimizer.step(); t_loss += loss.item()
    
    model.eval(); v_loss = 0; all_v_out, all_v_y = [], []
    with torch.no_grad():
        for vx, vr, vy in val_loader:
            vx, vr, vy = vx.to(device), vr.to(device), vy.to(device)
            v_out = model(vx, vr); v_loss += criterion(v_out, vy).item()
            all_v_out.append(v_out.cpu()); all_v_y.append(vy.cpu())
    
    avg_v_loss = v_loss / len(val_loader); scheduler.step(avg_v_loss)
    if epoch % 10 == 0:
        v_p = get_inverse_price(torch.cat(all_v_out).numpy(), scaler, len(FEATURES))
        v_a = get_inverse_price(torch.cat(all_v_y).numpy(), scaler, len(FEATURES))
        print(f"Epoch {epoch:>3} | Loss(T/V): {t_loss/len(train_loader):.4f}/{avg_v_loss:.4f} | {calculate_metrics(v_a, v_p)}")

    if avg_v_loss < best_val_loss:
        best_val_loss = avg_v_loss; torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best_transformer_sample.pth')); early_stop_cnt = 0
    else: early_stop_cnt += 1
    if early_stop_cnt >= 20: break

# 7. 최종 결과
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_transformer_sample.pth'))); model.eval(); all_p, all_a = [], []
with torch.no_grad():
    for tx, tr, ty in test_loader:
        all_p.append(model(tx.to(device), tr.to(device)).cpu()); all_a.append(ty.cpu())
y_p = get_inverse_price(torch.cat(all_p).numpy(), scaler, len(FEATURES))
y_a = get_inverse_price(torch.cat(all_a).numpy(), scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: Transformer (SAMPLE-WISE SPLIT)
Epoch  10 | Loss(T/V): 0.0127/0.0166 | R2: 0.9899 | MAE: 1,043 | RMSE: 1,592 | MAPE: 4.36% | MdAPE: 3.18% | RMSLE: 0.0591
Epoch  20 | Loss(T/V): 0.0105/0.0100 | R2: 0.9940 | MAE: 797 | RMSE: 1,227 | MAPE: 2.82% | MdAPE: 2.29% | RMSLE: 0.0357
Epoch  30 | Loss(T/V): 0.0100/0.0091 | R2: 0.9946 | MAE: 819 | RMSE: 1,162 | MAPE: 2.86% | MdAPE: 2.54% | RMSLE: 0.0342
Epoch  40 | Loss(T/V): 0.0085/0.0126 | R2: 0.9925 | MAE: 701 | RMSE: 1,373 | MAPE: 2.31% | MdAPE: 1.77% | RMSLE: 0.0319
------------------------------
FINAL TEST RESULT: R2: 0.9769 | MAE: 1,611 | RMSE: 2,314 | MAPE: 6.51% | MdAPE: 4.98% | RMSLE: 0.0805


### 대전

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings
import math

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리 (결측치 행 삭제)
def load_and_prepare():
    data_path = os.path.join(DATA_DIR, 'sido_대전광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    # 결측치가 있는 행 삭제 (실거래 데이터만 남김)
    df = df.dropna(subset=FEATURES)
    
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
n = len(train_df) + len(val_df) + len(test_df)
if n == 0:
    print("경고: 윈도우가 생성되지 않았습니다.")

scaler = StandardScaler()
# Train 데이터 기반으로 스케일러 학습
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class MultiModalDataset(Dataset):
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        self.x_seq = torch.FloatTensor(scaler.transform(x_raw.reshape(-1, F)).reshape(N, W, F))
        self.x_reg = torch.LongTensor(data_df['region_idx'].values)
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = torch.FloatTensor(scaler.transform(dummy)[:, 0].reshape(-1, 1))
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x_seq[idx], self.x_reg[idx], self.y[idx]

train_loader = DataLoader(MultiModalDataset(train_df, scaler, len(FEATURES)), batch_size=256, shuffle=True)
val_loader = DataLoader(MultiModalDataset(val_df, scaler, len(FEATURES)), batch_size=256)
test_loader = DataLoader(MultiModalDataset(test_df, scaler, len(FEATURES)), batch_size=256)

# 5. 모델 정의
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=50):
        super().__init__()
        pe = torch.zeros(max_len, d_model) # 일단 50행(시간) × 128열(특징) 크기의 0으로 채워진 빈 지도를 만듦
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1) # 0, 1, 2, ..., 49라는 숫자를 만들고(arange), 이를 세로 기둥 모양(unsqueeze(1))으로 세움. 이것이 각 데이터의 순번이 됨
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)) # 위치 인코딩(Positional Encoding) 공식의 구현체
        pe[:, 0::2] = torch.sin(position * div_term) # 짝수 열
        pe[:, 1::2] = torch.cos(position * div_term) # 홀수 열
        self.register_buffer('pe', pe.unsqueeze(0)) # 가중치가 아니니까 업데이트하지 말고, 모델 저장할 때 같이 저장만
    def forward(self, x): return x + self.pe[:, :x.size(1)]

class MultiRegionTransformer(nn.Module):
    def __init__(self, n_regions, n_features, d_model=128, nhead=8, num_layers=2, emb_dim=16):
        super().__init__()
        self.region_emb = nn.Embedding(n_regions, emb_dim) # 숫자로 된 지역 번호를 16차원의 밀집된 벡터로 변환
        self.feature_emb = nn.Linear(n_features, d_model) # 6개의 아파트 관련 특징(가격, 면적 등)을 128차원의 고차원으로 확장
        self.pos_encoder = PositionalEncoding(d_model) # 사인/코사인 파동을 더해 데이터에 시간 순서
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=256, dropout=0.1, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Sequential(nn.Linear(d_model + emb_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x_s, x_r):
        x = self.pos_encoder(self.feature_emb(x_s)) # 아파트 특징 데이터
        x = self.transformer_encoder(x)
        combined = torch.cat([x[:, -1, :], self.region_emb(x_r)], dim=1) # 앞선 1월부터 11월까지의 변화 양상을 모두 참고하여 업데이트된 현재 시장의 최종적인 특징과 지역 정보 결합
        return self.fc(combined)

model = MultiRegionTransformer(len(region_le.classes_), len(FEATURES)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001); criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 6. 학습
best_val_loss = float('inf'); early_stop_cnt = 0
print(f"{'='*30}\nSTART: Transformer (SAMPLE-WISE SPLIT)\n{'='*30}")

for epoch in range(1, 101):
    model.train(); t_loss = 0
    for xs, xr, y in train_loader:
        xs, xr, y = xs.to(device), xr.to(device), y.to(device)
        optimizer.zero_grad(); loss = criterion(model(xs, xr), y); loss.backward(); optimizer.step(); t_loss += loss.item()
    
    model.eval(); v_loss = 0; all_v_out, all_v_y = [], []
    with torch.no_grad():
        for vx, vr, vy in val_loader:
            vx, vr, vy = vx.to(device), vr.to(device), vy.to(device)
            v_out = model(vx, vr); v_loss += criterion(v_out, vy).item()
            all_v_out.append(v_out.cpu()); all_v_y.append(vy.cpu())
    
    avg_v_loss = v_loss / len(val_loader); scheduler.step(avg_v_loss)
    if epoch % 10 == 0:
        v_p = get_inverse_price(torch.cat(all_v_out).numpy(), scaler, len(FEATURES))
        v_a = get_inverse_price(torch.cat(all_v_y).numpy(), scaler, len(FEATURES))
        print(f"Epoch {epoch:>3} | Loss(T/V): {t_loss/len(train_loader):.4f}/{avg_v_loss:.4f} | {calculate_metrics(v_a, v_p)}")

    if avg_v_loss < best_val_loss:
        best_val_loss = avg_v_loss; torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best_transformer_sample.pth')); early_stop_cnt = 0
    else: early_stop_cnt += 1
    if early_stop_cnt >= 20: break

# 7. 최종 결과
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_transformer_sample.pth'))); model.eval(); all_p, all_a = [], []
with torch.no_grad():
    for tx, tr, ty in test_loader:
        all_p.append(model(tx.to(device), tr.to(device)).cpu()); all_a.append(ty.cpu())
y_p = get_inverse_price(torch.cat(all_p).numpy(), scaler, len(FEATURES))
y_a = get_inverse_price(torch.cat(all_a).numpy(), scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: Transformer (SAMPLE-WISE SPLIT)
Epoch  10 | Loss(T/V): 0.0200/0.0725 | R2: 0.9709 | MAE: 1,704 | RMSE: 2,984 | MAPE: 5.75% | MdAPE: 4.65% | RMSLE: 0.0752
Epoch  20 | Loss(T/V): 0.0165/0.0639 | R2: 0.9740 | MAE: 1,645 | RMSE: 2,818 | MAPE: 6.03% | MdAPE: 4.92% | RMSLE: 0.0823
Epoch  30 | Loss(T/V): 0.0164/0.0603 | R2: 0.9756 | MAE: 1,599 | RMSE: 2,733 | MAPE: 5.65% | MdAPE: 4.56% | RMSLE: 0.0757
Epoch  40 | Loss(T/V): 0.0152/0.0609 | R2: 0.9753 | MAE: 1,563 | RMSE: 2,751 | MAPE: 5.42% | MdAPE: 4.34% | RMSLE: 0.0736
------------------------------
FINAL TEST RESULT: R2: 0.8983 | MAE: 3,704 | RMSE: 6,378 | MAPE: 10.59% | MdAPE: 7.75% | RMSLE: 0.1358


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings
import math

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리
def load_and_prepare_2():
    data_path = os.path.join(DATA_DIR, 'sido_대전광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    for col in FEATURES:
        df[col] = df.groupby('sample_id')[col].transform(lambda x: x.interpolate(method='linear', limit_direction='both'))
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_2()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
n = len(train_df) + len(val_df) + len(test_df)
if n == 0:
    print("경고: 윈도우가 생성되지 않았습니다.")

scaler = StandardScaler()
# Train 데이터 기반으로 스케일러 학습
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class MultiModalDataset(Dataset):
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        self.x_seq = torch.FloatTensor(scaler.transform(x_raw.reshape(-1, F)).reshape(N, W, F))
        self.x_reg = torch.LongTensor(data_df['region_idx'].values)
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = torch.FloatTensor(scaler.transform(dummy)[:, 0].reshape(-1, 1))
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x_seq[idx], self.x_reg[idx], self.y[idx]

train_loader = DataLoader(MultiModalDataset(train_df, scaler, len(FEATURES)), batch_size=256, shuffle=True)
val_loader = DataLoader(MultiModalDataset(val_df, scaler, len(FEATURES)), batch_size=256)
test_loader = DataLoader(MultiModalDataset(test_df, scaler, len(FEATURES)), batch_size=256)

# 5. 모델 정의
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=50):
        super().__init__()
        pe = torch.zeros(max_len, d_model) # 일단 50행(시간) × 128열(특징) 크기의 0으로 채워진 빈 지도를 만듦
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1) # 0, 1, 2, ..., 49라는 숫자를 만들고(arange), 이를 세로 기둥 모양(unsqueeze(1))으로 세움. 이것이 각 데이터의 순번이 됨
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)) # 위치 인코딩(Positional Encoding) 공식의 구현체
        pe[:, 0::2] = torch.sin(position * div_term) # 짝수 열
        pe[:, 1::2] = torch.cos(position * div_term) # 홀수 열
        self.register_buffer('pe', pe.unsqueeze(0)) # 가중치가 아니니까 업데이트하지 말고, 모델 저장할 때 같이 저장만
    def forward(self, x): return x + self.pe[:, :x.size(1)]

class MultiRegionTransformer(nn.Module):
    def __init__(self, n_regions, n_features, d_model=128, nhead=8, num_layers=2, emb_dim=16):
        super().__init__()
        self.region_emb = nn.Embedding(n_regions, emb_dim) # 숫자로 된 지역 번호를 16차원의 밀집된 벡터로 변환
        self.feature_emb = nn.Linear(n_features, d_model) # 6개의 아파트 관련 특징(가격, 면적 등)을 128차원의 고차원으로 확장
        self.pos_encoder = PositionalEncoding(d_model) # 사인/코사인 파동을 더해 데이터에 시간 순서
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=256, dropout=0.1, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Sequential(nn.Linear(d_model + emb_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x_s, x_r):
        x = self.pos_encoder(self.feature_emb(x_s)) # 아파트 특징 데이터
        x = self.transformer_encoder(x)
        combined = torch.cat([x[:, -1, :], self.region_emb(x_r)], dim=1) # 앞선 1월부터 11월까지의 변화 양상을 모두 참고하여 업데이트된 현재 시장의 최종적인 특징과 지역 정보 결합
        return self.fc(combined)

model = MultiRegionTransformer(len(region_le.classes_), len(FEATURES)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001); criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 6. 학습
best_val_loss = float('inf'); early_stop_cnt = 0
print(f"{'='*30}\nSTART: Transformer (SAMPLE-WISE SPLIT)\n{'='*30}")

for epoch in range(1, 101):
    model.train(); t_loss = 0
    for xs, xr, y in train_loader:
        xs, xr, y = xs.to(device), xr.to(device), y.to(device)
        optimizer.zero_grad(); loss = criterion(model(xs, xr), y); loss.backward(); optimizer.step(); t_loss += loss.item()
    
    model.eval(); v_loss = 0; all_v_out, all_v_y = [], []
    with torch.no_grad():
        for vx, vr, vy in val_loader:
            vx, vr, vy = vx.to(device), vr.to(device), vy.to(device)
            v_out = model(vx, vr); v_loss += criterion(v_out, vy).item()
            all_v_out.append(v_out.cpu()); all_v_y.append(vy.cpu())
    
    avg_v_loss = v_loss / len(val_loader); scheduler.step(avg_v_loss)
    if epoch % 10 == 0:
        v_p = get_inverse_price(torch.cat(all_v_out).numpy(), scaler, len(FEATURES))
        v_a = get_inverse_price(torch.cat(all_v_y).numpy(), scaler, len(FEATURES))
        print(f"Epoch {epoch:>3} | Loss(T/V): {t_loss/len(train_loader):.4f}/{avg_v_loss:.4f} | {calculate_metrics(v_a, v_p)}")

    if avg_v_loss < best_val_loss:
        best_val_loss = avg_v_loss; torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best_transformer_sample.pth')); early_stop_cnt = 0
    else: early_stop_cnt += 1
    if early_stop_cnt >= 20: break

# 7. 최종 결과
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_transformer_sample.pth'))); model.eval(); all_p, all_a = [], []
with torch.no_grad():
    for tx, tr, ty in test_loader:
        all_p.append(model(tx.to(device), tr.to(device)).cpu()); all_a.append(ty.cpu())
y_p = get_inverse_price(torch.cat(all_p).numpy(), scaler, len(FEATURES))
y_a = get_inverse_price(torch.cat(all_a).numpy(), scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: Transformer (SAMPLE-WISE SPLIT)
Epoch  10 | Loss(T/V): 0.0155/0.0249 | R2: 0.9914 | MAE: 1,487 | RMSE: 2,117 | MAPE: 3.84% | MdAPE: 3.63% | RMSLE: 0.0436
Epoch  20 | Loss(T/V): 0.0127/0.0257 | R2: 0.9911 | MAE: 1,285 | RMSE: 2,156 | MAPE: 3.01% | MdAPE: 2.60% | RMSLE: 0.0370
Epoch  30 | Loss(T/V): 0.0115/0.0222 | R2: 0.9923 | MAE: 1,306 | RMSE: 2,001 | MAPE: 3.23% | MdAPE: 2.90% | RMSLE: 0.0394
------------------------------
FINAL TEST RESULT: R2: 0.9768 | MAE: 2,164 | RMSE: 3,328 | MAPE: 6.14% | MdAPE: 4.15% | RMSLE: 0.0794


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings
import math

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 고도화된 데이터 전처리 (하이브리드)
def load_and_prepare_advanced():
    data_path = os.path.join(DATA_DIR, 'sido_대전광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    
    rt = df.groupby(['region_id', 'transaction_date'])['price'].mean().reset_index().sort_values(['region_id', 'transaction_date'])
    rt['smooth_price'] = rt.groupby('region_id')['price'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
    rt['region_multiplier'] = 1 + rt.groupby('region_id')['smooth_price'].pct_change().fillna(0)
    df = pd.merge(df, rt[['region_id', 'transaction_date', 'region_multiplier']], on=['region_id', 'transaction_date'], how='left')
    
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    def apply_advanced_hybrid(group):
        group = group.sort_values('transaction_date')
        p_orig, mult = group['price'].values, group['region_multiplier'].values
        p_lin = group['price'].interpolate(method='linear', limit_direction='both').values
        actual_count = group['price'].notnull().sum()
        fidelity = actual_count / len(group)
        lin_weight = 0.8 + (0.15 * fidelity); reg_weight = 1.0 - lin_weight
        p_reg = p_orig.copy()
        for i in range(1, len(p_reg)):
            if np.isnan(p_reg[i]) and not np.isnan(p_reg[i-1]):
                p_reg[i] = p_reg[i-1] * (mult[i] if mult[i] != 0 else 1)
        p_reg = pd.Series(p_reg).fillna(pd.Series(p_lin)).values
        group['price'] = np.where(np.isnan(p_orig), (lin_weight * p_lin) + (reg_weight * p_reg), p_orig)
        for col in FEATURES[1:]: group[col] = group[col].interpolate(method='linear', limit_direction='both')
        return group
        
    df = df.groupby('sample_id').apply(apply_advanced_hybrid).reset_index(drop=True)
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_advanced()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
n = len(train_df) + len(val_df) + len(test_df)
if n == 0:
    print("경고: 윈도우가 생성되지 않았습니다.")

scaler = StandardScaler()
# Train 데이터 기반으로 스케일러 학습
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class MultiModalDataset(Dataset):
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        self.x_seq = torch.FloatTensor(scaler.transform(x_raw.reshape(-1, F)).reshape(N, W, F))
        self.x_reg = torch.LongTensor(data_df['region_idx'].values)
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = torch.FloatTensor(scaler.transform(dummy)[:, 0].reshape(-1, 1))
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x_seq[idx], self.x_reg[idx], self.y[idx]

train_loader = DataLoader(MultiModalDataset(train_df, scaler, len(FEATURES)), batch_size=256, shuffle=True)
val_loader = DataLoader(MultiModalDataset(val_df, scaler, len(FEATURES)), batch_size=256)
test_loader = DataLoader(MultiModalDataset(test_df, scaler, len(FEATURES)), batch_size=256)

# 5. 모델 정의
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=50):
        super().__init__()
        pe = torch.zeros(max_len, d_model) # 일단 50행(시간) × 128열(특징) 크기의 0으로 채워진 빈 지도를 만듦
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1) # 0, 1, 2, ..., 49라는 숫자를 만들고(arange), 이를 세로 기둥 모양(unsqueeze(1))으로 세움. 이것이 각 데이터의 순번이 됨
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)) # 위치 인코딩(Positional Encoding) 공식의 구현체
        pe[:, 0::2] = torch.sin(position * div_term) # 짝수 열
        pe[:, 1::2] = torch.cos(position * div_term) # 홀수 열
        self.register_buffer('pe', pe.unsqueeze(0)) # 가중치가 아니니까 업데이트하지 말고, 모델 저장할 때 같이 저장만
    def forward(self, x): return x + self.pe[:, :x.size(1)]

class MultiRegionTransformer(nn.Module):
    def __init__(self, n_regions, n_features, d_model=128, nhead=8, num_layers=2, emb_dim=16):
        super().__init__()
        self.region_emb = nn.Embedding(n_regions, emb_dim) # 숫자로 된 지역 번호를 16차원의 밀집된 벡터로 변환
        self.feature_emb = nn.Linear(n_features, d_model) # 6개의 아파트 관련 특징(가격, 면적 등)을 128차원의 고차원으로 확장
        self.pos_encoder = PositionalEncoding(d_model) # 사인/코사인 파동을 더해 데이터에 시간 순서
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=256, dropout=0.1, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Sequential(nn.Linear(d_model + emb_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x_s, x_r):
        x = self.pos_encoder(self.feature_emb(x_s)) # 아파트 특징 데이터
        x = self.transformer_encoder(x)
        combined = torch.cat([x[:, -1, :], self.region_emb(x_r)], dim=1) # 앞선 1월부터 11월까지의 변화 양상을 모두 참고하여 업데이트된 현재 시장의 최종적인 특징과 지역 정보 결합
        return self.fc(combined)

model = MultiRegionTransformer(len(region_le.classes_), len(FEATURES)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001); criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 6. 학습
best_val_loss = float('inf'); early_stop_cnt = 0
print(f"{'='*30}\nSTART: Transformer (SAMPLE-WISE SPLIT)\n{'='*30}")

for epoch in range(1, 101):
    model.train(); t_loss = 0
    for xs, xr, y in train_loader:
        xs, xr, y = xs.to(device), xr.to(device), y.to(device)
        optimizer.zero_grad(); loss = criterion(model(xs, xr), y); loss.backward(); optimizer.step(); t_loss += loss.item()
    
    model.eval(); v_loss = 0; all_v_out, all_v_y = [], []
    with torch.no_grad():
        for vx, vr, vy in val_loader:
            vx, vr, vy = vx.to(device), vr.to(device), vy.to(device)
            v_out = model(vx, vr); v_loss += criterion(v_out, vy).item()
            all_v_out.append(v_out.cpu()); all_v_y.append(vy.cpu())
    
    avg_v_loss = v_loss / len(val_loader); scheduler.step(avg_v_loss)
    if epoch % 10 == 0:
        v_p = get_inverse_price(torch.cat(all_v_out).numpy(), scaler, len(FEATURES))
        v_a = get_inverse_price(torch.cat(all_v_y).numpy(), scaler, len(FEATURES))
        print(f"Epoch {epoch:>3} | Loss(T/V): {t_loss/len(train_loader):.4f}/{avg_v_loss:.4f} | {calculate_metrics(v_a, v_p)}")

    if avg_v_loss < best_val_loss:
        best_val_loss = avg_v_loss; torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best_transformer_sample.pth')); early_stop_cnt = 0
    else: early_stop_cnt += 1
    if early_stop_cnt >= 20: break

# 7. 최종 결과
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_transformer_sample.pth'))); model.eval(); all_p, all_a = [], []
with torch.no_grad():
    for tx, tr, ty in test_loader:
        all_p.append(model(tx.to(device), tr.to(device)).cpu()); all_a.append(ty.cpu())
y_p = get_inverse_price(torch.cat(all_p).numpy(), scaler, len(FEATURES))
y_a = get_inverse_price(torch.cat(all_a).numpy(), scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: Transformer (SAMPLE-WISE SPLIT)
Epoch  10 | Loss(T/V): 0.0149/0.0327 | R2: 0.9882 | MAE: 1,701 | RMSE: 2,427 | MAPE: 4.47% | MdAPE: 4.18% | RMSLE: 0.0503
Epoch  20 | Loss(T/V): 0.0120/0.0235 | R2: 0.9914 | MAE: 1,306 | RMSE: 2,064 | MAPE: 3.29% | MdAPE: 2.83% | RMSLE: 0.0407
------------------------------
FINAL TEST RESULT: R2: 0.9807 | MAE: 1,833 | RMSE: 3,001 | MAPE: 5.35% | MdAPE: 3.59% | RMSLE: 0.0718


### 광주

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings
import math

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리 (결측치 행 삭제)
def load_and_prepare():
    data_path = os.path.join(DATA_DIR, 'sido_광주광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    # 결측치가 있는 행 삭제 (실거래 데이터만 남김)
    df = df.dropna(subset=FEATURES)
    
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
n = len(train_df) + len(val_df) + len(test_df)
if n == 0:
    print("경고: 윈도우가 생성되지 않았습니다.")

scaler = StandardScaler()
# Train 데이터 기반으로 스케일러 학습
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class MultiModalDataset(Dataset):
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        self.x_seq = torch.FloatTensor(scaler.transform(x_raw.reshape(-1, F)).reshape(N, W, F))
        self.x_reg = torch.LongTensor(data_df['region_idx'].values)
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = torch.FloatTensor(scaler.transform(dummy)[:, 0].reshape(-1, 1))
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x_seq[idx], self.x_reg[idx], self.y[idx]

train_loader = DataLoader(MultiModalDataset(train_df, scaler, len(FEATURES)), batch_size=256, shuffle=True)
val_loader = DataLoader(MultiModalDataset(val_df, scaler, len(FEATURES)), batch_size=256)
test_loader = DataLoader(MultiModalDataset(test_df, scaler, len(FEATURES)), batch_size=256)

# 5. 모델 정의
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=50):
        super().__init__()
        pe = torch.zeros(max_len, d_model) # 일단 50행(시간) × 128열(특징) 크기의 0으로 채워진 빈 지도를 만듦
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1) # 0, 1, 2, ..., 49라는 숫자를 만들고(arange), 이를 세로 기둥 모양(unsqueeze(1))으로 세움. 이것이 각 데이터의 순번이 됨
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)) # 위치 인코딩(Positional Encoding) 공식의 구현체
        pe[:, 0::2] = torch.sin(position * div_term) # 짝수 열
        pe[:, 1::2] = torch.cos(position * div_term) # 홀수 열
        self.register_buffer('pe', pe.unsqueeze(0)) # 가중치가 아니니까 업데이트하지 말고, 모델 저장할 때 같이 저장만
    def forward(self, x): return x + self.pe[:, :x.size(1)]

class MultiRegionTransformer(nn.Module):
    def __init__(self, n_regions, n_features, d_model=128, nhead=8, num_layers=2, emb_dim=16):
        super().__init__()
        self.region_emb = nn.Embedding(n_regions, emb_dim) # 숫자로 된 지역 번호를 16차원의 밀집된 벡터로 변환
        self.feature_emb = nn.Linear(n_features, d_model) # 6개의 아파트 관련 특징(가격, 면적 등)을 128차원의 고차원으로 확장
        self.pos_encoder = PositionalEncoding(d_model) # 사인/코사인 파동을 더해 데이터에 시간 순서
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=256, dropout=0.1, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Sequential(nn.Linear(d_model + emb_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x_s, x_r):
        x = self.pos_encoder(self.feature_emb(x_s)) # 아파트 특징 데이터
        x = self.transformer_encoder(x)
        combined = torch.cat([x[:, -1, :], self.region_emb(x_r)], dim=1) # 앞선 1월부터 11월까지의 변화 양상을 모두 참고하여 업데이트된 현재 시장의 최종적인 특징과 지역 정보 결합
        return self.fc(combined)

model = MultiRegionTransformer(len(region_le.classes_), len(FEATURES)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001); criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 6. 학습
best_val_loss = float('inf'); early_stop_cnt = 0
print(f"{'='*30}\nSTART: Transformer (SAMPLE-WISE SPLIT)\n{'='*30}")

for epoch in range(1, 101):
    model.train(); t_loss = 0
    for xs, xr, y in train_loader:
        xs, xr, y = xs.to(device), xr.to(device), y.to(device)
        optimizer.zero_grad(); loss = criterion(model(xs, xr), y); loss.backward(); optimizer.step(); t_loss += loss.item()
    
    model.eval(); v_loss = 0; all_v_out, all_v_y = [], []
    with torch.no_grad():
        for vx, vr, vy in val_loader:
            vx, vr, vy = vx.to(device), vr.to(device), vy.to(device)
            v_out = model(vx, vr); v_loss += criterion(v_out, vy).item()
            all_v_out.append(v_out.cpu()); all_v_y.append(vy.cpu())
    
    avg_v_loss = v_loss / len(val_loader); scheduler.step(avg_v_loss)
    if epoch % 10 == 0:
        v_p = get_inverse_price(torch.cat(all_v_out).numpy(), scaler, len(FEATURES))
        v_a = get_inverse_price(torch.cat(all_v_y).numpy(), scaler, len(FEATURES))
        print(f"Epoch {epoch:>3} | Loss(T/V): {t_loss/len(train_loader):.4f}/{avg_v_loss:.4f} | {calculate_metrics(v_a, v_p)}")

    if avg_v_loss < best_val_loss:
        best_val_loss = avg_v_loss; torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best_transformer_sample.pth')); early_stop_cnt = 0
    else: early_stop_cnt += 1
    if early_stop_cnt >= 20: break

# 7. 최종 결과
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_transformer_sample.pth'))); model.eval(); all_p, all_a = [], []
with torch.no_grad():
    for tx, tr, ty in test_loader:
        all_p.append(model(tx.to(device), tr.to(device)).cpu()); all_a.append(ty.cpu())
y_p = get_inverse_price(torch.cat(all_p).numpy(), scaler, len(FEATURES))
y_a = get_inverse_price(torch.cat(all_a).numpy(), scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: Transformer (SAMPLE-WISE SPLIT)
Epoch  10 | Loss(T/V): 0.0210/0.0287 | R2: 0.9825 | MAE: 1,058 | RMSE: 1,548 | MAPE: 5.45% | MdAPE: 4.38% | RMSLE: 0.0716
Epoch  20 | Loss(T/V): 0.0164/0.0263 | R2: 0.9840 | MAE: 954 | RMSE: 1,480 | MAPE: 4.86% | MdAPE: 3.80% | RMSLE: 0.0658
------------------------------
FINAL TEST RESULT: R2: 0.9124 | MAE: 2,609 | RMSE: 4,148 | MAPE: 10.27% | MdAPE: 8.20% | RMSLE: 0.1447


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings
import math

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리
def load_and_prepare_2():
    data_path = os.path.join(DATA_DIR, 'sido_광주광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    for col in FEATURES:
        df[col] = df.groupby('sample_id')[col].transform(lambda x: x.interpolate(method='linear', limit_direction='both'))
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_2()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
n = len(train_df) + len(val_df) + len(test_df)
if n == 0:
    print("경고: 윈도우가 생성되지 않았습니다.")

scaler = StandardScaler()
# Train 데이터 기반으로 스케일러 학습
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class MultiModalDataset(Dataset):
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        self.x_seq = torch.FloatTensor(scaler.transform(x_raw.reshape(-1, F)).reshape(N, W, F))
        self.x_reg = torch.LongTensor(data_df['region_idx'].values)
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = torch.FloatTensor(scaler.transform(dummy)[:, 0].reshape(-1, 1))
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x_seq[idx], self.x_reg[idx], self.y[idx]

train_loader = DataLoader(MultiModalDataset(train_df, scaler, len(FEATURES)), batch_size=256, shuffle=True)
val_loader = DataLoader(MultiModalDataset(val_df, scaler, len(FEATURES)), batch_size=256)
test_loader = DataLoader(MultiModalDataset(test_df, scaler, len(FEATURES)), batch_size=256)

# 5. 모델 정의
# 5. 모델 정의
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=50):
        super().__init__()
        pe = torch.zeros(max_len, d_model) # 일단 50행(시간) × 128열(특징) 크기의 0으로 채워진 빈 지도를 만듦
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1) # 0, 1, 2, ..., 49라는 숫자를 만들고(arange), 이를 세로 기둥 모양(unsqueeze(1))으로 세움. 이것이 각 데이터의 순번이 됨
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)) # 위치 인코딩(Positional Encoding) 공식의 구현체
        pe[:, 0::2] = torch.sin(position * div_term) # 짝수 열
        pe[:, 1::2] = torch.cos(position * div_term) # 홀수 열
        self.register_buffer('pe', pe.unsqueeze(0)) # 가중치가 아니니까 업데이트하지 말고, 모델 저장할 때 같이 저장만
    def forward(self, x): return x + self.pe[:, :x.size(1)]

class MultiRegionTransformer(nn.Module):
    def __init__(self, n_regions, n_features, d_model=128, nhead=8, num_layers=2, emb_dim=16):
        super().__init__()
        self.region_emb = nn.Embedding(n_regions, emb_dim) # 숫자로 된 지역 번호를 16차원의 밀집된 벡터로 변환
        self.feature_emb = nn.Linear(n_features, d_model) # 6개의 아파트 관련 특징(가격, 면적 등)을 128차원의 고차원으로 확장
        self.pos_encoder = PositionalEncoding(d_model) # 사인/코사인 파동을 더해 데이터에 시간 순서
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=256, dropout=0.1, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Sequential(nn.Linear(d_model + emb_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x_s, x_r):
        x = self.pos_encoder(self.feature_emb(x_s)) # 아파트 특징 데이터
        x = self.transformer_encoder(x)
        combined = torch.cat([x[:, -1, :], self.region_emb(x_r)], dim=1) # 앞선 1월부터 11월까지의 변화 양상을 모두 참고하여 업데이트된 현재 시장의 최종적인 특징과 지역 정보 결합
        return self.fc(combined)

model = MultiRegionTransformer(len(region_le.classes_), len(FEATURES)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001); criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 6. 학습
best_val_loss = float('inf'); early_stop_cnt = 0
print(f"{'='*30}\nSTART: Transformer (SAMPLE-WISE SPLIT)\n{'='*30}")

for epoch in range(1, 101):
    model.train(); t_loss = 0
    for xs, xr, y in train_loader:
        xs, xr, y = xs.to(device), xr.to(device), y.to(device)
        optimizer.zero_grad(); loss = criterion(model(xs, xr), y); loss.backward(); optimizer.step(); t_loss += loss.item()
    
    model.eval(); v_loss = 0; all_v_out, all_v_y = [], []
    with torch.no_grad():
        for vx, vr, vy in val_loader:
            vx, vr, vy = vx.to(device), vr.to(device), vy.to(device)
            v_out = model(vx, vr); v_loss += criterion(v_out, vy).item()
            all_v_out.append(v_out.cpu()); all_v_y.append(vy.cpu())
    
    avg_v_loss = v_loss / len(val_loader); scheduler.step(avg_v_loss)
    if epoch % 10 == 0:
        v_p = get_inverse_price(torch.cat(all_v_out).numpy(), scaler, len(FEATURES))
        v_a = get_inverse_price(torch.cat(all_v_y).numpy(), scaler, len(FEATURES))
        print(f"Epoch {epoch:>3} | Loss(T/V): {t_loss/len(train_loader):.4f}/{avg_v_loss:.4f} | {calculate_metrics(v_a, v_p)}")

    if avg_v_loss < best_val_loss:
        best_val_loss = avg_v_loss; torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best_transformer_sample.pth')); early_stop_cnt = 0
    else: early_stop_cnt += 1
    if early_stop_cnt >= 20: break

# 7. 최종 결과
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_transformer_sample.pth'))); model.eval(); all_p, all_a = [], []
with torch.no_grad():
    for tx, tr, ty in test_loader:
        all_p.append(model(tx.to(device), tr.to(device)).cpu()); all_a.append(ty.cpu())
y_p = get_inverse_price(torch.cat(all_p).numpy(), scaler, len(FEATURES))
y_a = get_inverse_price(torch.cat(all_a).numpy(), scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: Transformer (SAMPLE-WISE SPLIT)
Epoch  10 | Loss(T/V): 0.0130/0.0093 | R2: 0.9957 | MAE: 590 | RMSE: 1,007 | MAPE: 2.81% | MdAPE: 2.20% | RMSLE: 0.0376
Epoch  20 | Loss(T/V): 0.0111/0.0090 | R2: 0.9958 | MAE: 572 | RMSE: 991 | MAPE: 2.51% | MdAPE: 1.99% | RMSLE: 0.0333
Epoch  30 | Loss(T/V): 0.0102/0.0129 | R2: 0.9941 | MAE: 733 | RMSE: 1,184 | MAPE: 2.96% | MdAPE: 2.54% | RMSLE: 0.0372
Epoch  40 | Loss(T/V): 0.0096/0.0093 | R2: 0.9957 | MAE: 609 | RMSE: 1,005 | MAPE: 2.57% | MdAPE: 2.14% | RMSLE: 0.0335
------------------------------
FINAL TEST RESULT: R2: 0.9722 | MAE: 1,820 | RMSE: 2,662 | MAPE: 7.56% | MdAPE: 5.97% | RMSLE: 0.0926


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings
import math

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 고도화된 데이터 전처리 (하이브리드)
def load_and_prepare_advanced():
    data_path = os.path.join(DATA_DIR, 'sido_광주광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    
    rt = df.groupby(['region_id', 'transaction_date'])['price'].mean().reset_index().sort_values(['region_id', 'transaction_date'])
    rt['smooth_price'] = rt.groupby('region_id')['price'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
    rt['region_multiplier'] = 1 + rt.groupby('region_id')['smooth_price'].pct_change().fillna(0)
    df = pd.merge(df, rt[['region_id', 'transaction_date', 'region_multiplier']], on=['region_id', 'transaction_date'], how='left')
    
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    def apply_advanced_hybrid(group):
        group = group.sort_values('transaction_date')
        p_orig, mult = group['price'].values, group['region_multiplier'].values
        p_lin = group['price'].interpolate(method='linear', limit_direction='both').values
        actual_count = group['price'].notnull().sum()
        fidelity = actual_count / len(group)
        lin_weight = 0.8 + (0.15 * fidelity); reg_weight = 1.0 - lin_weight
        p_reg = p_orig.copy()
        for i in range(1, len(p_reg)):
            if np.isnan(p_reg[i]) and not np.isnan(p_reg[i-1]):
                p_reg[i] = p_reg[i-1] * (mult[i] if mult[i] != 0 else 1)
        p_reg = pd.Series(p_reg).fillna(pd.Series(p_lin)).values
        group['price'] = np.where(np.isnan(p_orig), (lin_weight * p_lin) + (reg_weight * p_reg), p_orig)
        for col in FEATURES[1:]: group[col] = group[col].interpolate(method='linear', limit_direction='both')
        return group
        
    df = df.groupby('sample_id').apply(apply_advanced_hybrid).reset_index(drop=True)
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_advanced()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
n = len(train_df) + len(val_df) + len(test_df)
if n == 0:
    print("경고: 윈도우가 생성되지 않았습니다.")

scaler = StandardScaler()
# Train 데이터 기반으로 스케일러 학습
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class MultiModalDataset(Dataset):
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        self.x_seq = torch.FloatTensor(scaler.transform(x_raw.reshape(-1, F)).reshape(N, W, F))
        self.x_reg = torch.LongTensor(data_df['region_idx'].values)
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = torch.FloatTensor(scaler.transform(dummy)[:, 0].reshape(-1, 1))
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x_seq[idx], self.x_reg[idx], self.y[idx]

train_loader = DataLoader(MultiModalDataset(train_df, scaler, len(FEATURES)), batch_size=256, shuffle=True)
val_loader = DataLoader(MultiModalDataset(val_df, scaler, len(FEATURES)), batch_size=256)
test_loader = DataLoader(MultiModalDataset(test_df, scaler, len(FEATURES)), batch_size=256)

# 5. 모델 정의
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=50):
        super().__init__()
        pe = torch.zeros(max_len, d_model) # 일단 50행(시간) × 128열(특징) 크기의 0으로 채워진 빈 지도를 만듦
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1) # 0, 1, 2, ..., 49라는 숫자를 만들고(arange), 이를 세로 기둥 모양(unsqueeze(1))으로 세움. 이것이 각 데이터의 순번이 됨
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)) # 위치 인코딩(Positional Encoding) 공식의 구현체
        pe[:, 0::2] = torch.sin(position * div_term) # 짝수 열
        pe[:, 1::2] = torch.cos(position * div_term) # 홀수 열
        self.register_buffer('pe', pe.unsqueeze(0)) # 가중치가 아니니까 업데이트하지 말고, 모델 저장할 때 같이 저장만
    def forward(self, x): return x + self.pe[:, :x.size(1)]

class MultiRegionTransformer(nn.Module):
    def __init__(self, n_regions, n_features, d_model=128, nhead=8, num_layers=2, emb_dim=16):
        super().__init__()
        self.region_emb = nn.Embedding(n_regions, emb_dim) # 숫자로 된 지역 번호를 16차원의 밀집된 벡터로 변환
        self.feature_emb = nn.Linear(n_features, d_model) # 6개의 아파트 관련 특징(가격, 면적 등)을 128차원의 고차원으로 확장
        self.pos_encoder = PositionalEncoding(d_model) # 사인/코사인 파동을 더해 데이터에 시간 순서
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=256, dropout=0.1, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Sequential(nn.Linear(d_model + emb_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x_s, x_r):
        x = self.pos_encoder(self.feature_emb(x_s)) # 아파트 특징 데이터
        x = self.transformer_encoder(x)
        combined = torch.cat([x[:, -1, :], self.region_emb(x_r)], dim=1) # 앞선 1월부터 11월까지의 변화 양상을 모두 참고하여 업데이트된 현재 시장의 최종적인 특징과 지역 정보 결합
        return self.fc(combined)

model = MultiRegionTransformer(len(region_le.classes_), len(FEATURES)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001); criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 6. 학습
best_val_loss = float('inf'); early_stop_cnt = 0
print(f"{'='*30}\nSTART: Transformer (SAMPLE-WISE SPLIT)\n{'='*30}")

for epoch in range(1, 101):
    model.train(); t_loss = 0
    for xs, xr, y in train_loader:
        xs, xr, y = xs.to(device), xr.to(device), y.to(device)
        optimizer.zero_grad(); loss = criterion(model(xs, xr), y); loss.backward(); optimizer.step(); t_loss += loss.item()
    
    model.eval(); v_loss = 0; all_v_out, all_v_y = [], []
    with torch.no_grad():
        for vx, vr, vy in val_loader:
            vx, vr, vy = vx.to(device), vr.to(device), vy.to(device)
            v_out = model(vx, vr); v_loss += criterion(v_out, vy).item()
            all_v_out.append(v_out.cpu()); all_v_y.append(vy.cpu())
    
    avg_v_loss = v_loss / len(val_loader); scheduler.step(avg_v_loss)
    if epoch % 10 == 0:
        v_p = get_inverse_price(torch.cat(all_v_out).numpy(), scaler, len(FEATURES))
        v_a = get_inverse_price(torch.cat(all_v_y).numpy(), scaler, len(FEATURES))
        print(f"Epoch {epoch:>3} | Loss(T/V): {t_loss/len(train_loader):.4f}/{avg_v_loss:.4f} | {calculate_metrics(v_a, v_p)}")

    if avg_v_loss < best_val_loss:
        best_val_loss = avg_v_loss; torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best_transformer_sample.pth')); early_stop_cnt = 0
    else: early_stop_cnt += 1
    if early_stop_cnt >= 20: break

# 7. 최종 결과
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_transformer_sample.pth'))); model.eval(); all_p, all_a = [], []
with torch.no_grad():
    for tx, tr, ty in test_loader:
        all_p.append(model(tx.to(device), tr.to(device)).cpu()); all_a.append(ty.cpu())
y_p = get_inverse_price(torch.cat(all_p).numpy(), scaler, len(FEATURES))
y_a = get_inverse_price(torch.cat(all_a).numpy(), scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: Transformer (SAMPLE-WISE SPLIT)
Epoch  10 | Loss(T/V): 0.0130/0.0081 | R2: 0.9961 | MAE: 574 | RMSE: 937 | MAPE: 2.75% | MdAPE: 2.23% | RMSLE: 0.0363
Epoch  20 | Loss(T/V): 0.0112/0.0076 | R2: 0.9963 | MAE: 587 | RMSE: 909 | MAPE: 2.79% | MdAPE: 2.19% | RMSLE: 0.0367
Epoch  30 | Loss(T/V): 0.0103/0.0122 | R2: 0.9941 | MAE: 743 | RMSE: 1,151 | MAPE: 3.04% | MdAPE: 2.63% | RMSLE: 0.0380
Epoch  40 | Loss(T/V): 0.0096/0.0084 | R2: 0.9959 | MAE: 607 | RMSE: 958 | MAPE: 2.58% | MdAPE: 2.10% | RMSLE: 0.0333
------------------------------
FINAL TEST RESULT: R2: 0.9757 | MAE: 1,667 | RMSE: 2,472 | MAPE: 7.13% | MdAPE: 5.51% | RMSLE: 0.0891
